# 05 — Request-bound release closure, binding ablation, and numerical sensitivity

This notebook is a targeted follow-up to the immutable audited-v4
00–04 evidence. Run it first with `CLOSURE_MODE = "pilot"`; change
only that value to `"full"` after the pilot reports PASS.

The notebook deliberately fails if the audited full evidence,
pinned code, or actual DuckDB database is missing or modified. It
contains no synthetic-data or synthetic-answer fallback.


In [1]:
# Change only CLOSURE_MODE after the pilot passes.
CLOSURE_MODE = "full"  # allowed: "pilot", "full"
ENVIRONMENT_LABEL = "colab_cpu_primary"

import subprocess
import sys
import importlib.metadata

required_packages = {
    "duckdb": "1.3.2",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "osqp": "1.0.4",
    "sqlglot": "27.0.0",
    "pyarrow": "19.0.1",
}
mismatched_packages = []
for package, required_version in required_packages.items():
    try:
        observed_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        observed_version = "missing"
    if observed_version != required_version:
        mismatched_packages.append(f"{package}=={required_version}")
if mismatched_packages:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            *mismatched_packages,
        ]
    )
else:
    print("Pinned dependencies already available; installation skipped.")
for package, required_version in required_packages.items():
    observed_version = importlib.metadata.version(package)
    if observed_version != required_version:
        raise RuntimeError(
            f"Dependency version mismatch after installation: {package} "
            f"{observed_version} != {required_version}"
        )

from pathlib import Path
import hashlib
import importlib
import json
import os
import re
import shutil
import tempfile
import time
import zipfile

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True
if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    default_root = "/content/drive/MyDrive/Arbiter_ICDE_2027"
else:
    default_root = "/content/Arbiter_ICDE_2027"

PROJECT_ROOT = Path(os.environ.get("ARBITER_PROJECT_ROOT", default_root)).resolve()
active_path = PROJECT_ROOT / "active_run.json"
if not active_path.is_file():
    raise FileNotFoundError(
        "The audited full run is not active. Run audited Notebooks 00-04 "
        "in full mode before this closure notebook."
    )
active = json.loads(active_path.read_text(encoding="utf-8"))
if active.get("run_mode") != "full":
    raise RuntimeError("The closure notebook requires the audited full run.")
if active.get("suite_version") != "2026-08-01-audited-v4":
    raise RuntimeError("The active input is not audited-v4.")
ACTIVE_RUN_DIR = (PROJECT_ROOT / active["run_directory"]).resolve()
if PROJECT_ROOT not in ACTIVE_RUN_DIR.parents:
    raise RuntimeError("The active run path escaped the project root.")

# Notebook 04 deliberately writes its immutable, checksummed evidence into
# a nested package directory. Resolve that package instead of treating the
# mutable active-run root as if it contained the package manifest.
evidence_package_name = (
    f"arbiter_icde_evidence_{active['run_mode']}_audited_v4"
)
required_markers = (
    "run_config.json",
    "checksums.csv",
    "checksums.sha256",
    "04_evidence/all_stage_quality_checks.csv",
    "code/arbiter_core.py",
)

def is_evidence_package(path):
    return path.is_dir() and all((path / marker).is_file() for marker in required_markers)

nested_evidence_dir = ACTIVE_RUN_DIR / evidence_package_name
if is_evidence_package(nested_evidence_dir):
    EVIDENCE_DIR = nested_evidence_dir.resolve()
    EVIDENCE_SOURCE = "nested_notebook04_package"
elif is_evidence_package(ACTIVE_RUN_DIR):
    # Retain compatibility with an explicitly extracted package used as an
    # active input, while preferring the real Colab layout above.
    EVIDENCE_DIR = ACTIVE_RUN_DIR
    EVIDENCE_SOURCE = "direct_extracted_package"
else:
    evidence_zip = ACTIVE_RUN_DIR / f"{evidence_package_name}.zip"
    if not evidence_zip.is_file():
        missing = [
            str(nested_evidence_dir / marker)
            for marker in required_markers
            if not (nested_evidence_dir / marker).is_file()
        ]
        raise FileNotFoundError(
            "Notebook 04's audited evidence package is incomplete. "
            "Do not create checksums manually. Rerun audited Notebook 04 "
            "in full mode. First missing marker: "
            + (missing[0] if missing else str(nested_evidence_dir))
        )

    extraction_parent = Path(
        tempfile.mkdtemp(prefix="arbiter-audited-v4-evidence-")
    )
    expected_prefix = evidence_package_name + "/"
    with zipfile.ZipFile(evidence_zip) as archive:
        infos = archive.infolist()
        names = [info.filename for info in infos]
        if len(names) != len(set(names)):
            raise RuntimeError("The audited evidence ZIP contains duplicate members.")
        if len(infos) > 10000:
            raise RuntimeError("The audited evidence ZIP has an unexpected member count.")
        if sum(info.file_size for info in infos) > 2 * 1024**3:
            raise RuntimeError("The audited evidence ZIP exceeds the 2 GiB safety limit.")
        for info in infos:
            member = Path(info.filename)
            mode = (info.external_attr >> 16) & 0o170000
            if (
                info.flag_bits & 0x1
                or info.filename.startswith(("/", "\\"))
                or "\\" in info.filename
                or member.is_absolute()
                or ".." in member.parts
                or not info.filename.startswith(expected_prefix)
                or mode == 0o120000
            ):
                raise RuntimeError(
                    f"Unsafe audited evidence ZIP member: {info.filename}"
                )
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt audited evidence ZIP member: {bad_member}")
        archive.extractall(extraction_parent)
    EVIDENCE_DIR = (extraction_parent / evidence_package_name).resolve()
    if not is_evidence_package(EVIDENCE_DIR):
        raise RuntimeError("The extracted audited evidence package is incomplete.")
    EVIDENCE_SOURCE = "verified_zip_extraction"

CODE_DIR = EVIDENCE_DIR / "code"
if not CODE_DIR.is_dir():
    raise FileNotFoundError("The pinned audited-v4 source directory is missing.")

closure_mode_config = {
    "pilot": {
        "concurrency_trials": 3,
        "budget_repetitions": 1,
        "budget_warmups": 0,
        "relative_tolerances": [1e-8, 1e-6],
        "margin_ratios": [3e-8, 3e-6],
        "overhead_repetitions": 3,
        "overhead_warmups": 0,
    },
    "full": {
        "concurrency_trials": 50,
        "budget_repetitions": 5,
        "budget_warmups": 2,
        "relative_tolerances": [1e-10, 1e-8, 1e-6, 1e-4],
        "margin_ratios": [3e-10, 3e-8, 3e-6, 3e-4],
        "overhead_repetitions": 25,
        "overhead_warmups": 5,
    },
}
if CLOSURE_MODE not in closure_mode_config:
    raise ValueError("CLOSURE_MODE must be 'pilot' or 'full'.")
mode_options = closure_mode_config[CLOSURE_MODE]

# Materialize the exact embedded closure modules before importing them.
local_code_dir = Path(
    tempfile.mkdtemp(prefix="arbiter-request-bound-closure-code-")
)
embedded_sources = {
    "arbiter_request_release.py": '"""Request-bound release protocol for the ARBITER closure campaign.\n\nThis module is intentionally separate from the audited-v4 implementation.  It\ndoes not change the immutable 00--04 evidence.  It closes one specific API\ndefect: a token must authorize exactly one request, buyer/session, asset, quote,\nquery, and immutable answer buffer under one verified constraint snapshot.\n\nThe durable boundary is a local SQLite transaction.  A successful commit\ninserts a purchase authorization and changes the staged answer to ``released``\nin the same transaction.  Subsequent network delivery is recoverable from the\noutbox but is outside this in-process prototype\'s atomicity claim.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, replace\nimport base64\nimport datetime as _datetime\nfrom decimal import Decimal\nimport hashlib\nimport hmac\nimport json\nimport math\nfrom pathlib import Path\nimport secrets\nimport sqlite3\nimport threading\nimport time\nfrom typing import Any, Callable, Mapping\n\nimport numpy as np\n\nimport arbiter_core as _core\n\n\nPROTOCOL_VERSION = 2\nPROTOCOL_PURPOSE = "arbiter.request_bound_release"\nDEFAULT_TOKEN_TTL_NS = 5 * 60 * 1_000_000_000\nMAX_ID_UTF8_BYTES = 512\nMAX_SQL_UTF8_BYTES = 4 * 1024 * 1024\n\n\ndef _sha256_hex(*parts: bytes) -> str:\n    digest = hashlib.sha256()\n    for part in parts:\n        digest.update(part)\n    return digest.hexdigest()\n\n\ndef _canonical_json(value: Any) -> bytes:\n    return json.dumps(\n        value,\n        sort_keys=True,\n        separators=(",", ":"),\n        ensure_ascii=True,\n        allow_nan=False,\n    ).encode("utf-8")\n\n\ndef _validated_identifier(name: str, value: Any) -> str:\n    if not isinstance(value, str):\n        raise TypeError(f"{name} must be a string")\n    if not value or value.strip() != value:\n        raise ValueError(f"{name} must be nonempty and have no edge whitespace")\n    encoded = value.encode("utf-8")\n    if len(encoded) > MAX_ID_UTF8_BYTES:\n        raise ValueError(f"{name} exceeds {MAX_ID_UTF8_BYTES} UTF-8 bytes")\n    if any(ord(character) < 32 for character in value):\n        raise ValueError(f"{name} contains a control character")\n    return value\n\n\ndef _validated_digest(name: str, value: Any) -> str:\n    digest = _validated_identifier(name, value)\n    if len(digest) != 64 or any(character not in "0123456789abcdef" for character in digest):\n        raise ValueError(f"{name} must be a lowercase SHA-256 hex digest")\n    return digest\n\n\ndef _validated_tolerance(value: Any) -> float:\n    tolerance = float(value)\n    if not math.isfinite(tolerance) or tolerance <= 0.0:\n        raise ValueError("verifier tolerance must be finite and strictly positive")\n    return tolerance\n\n\ndef _float64_hex(value: Any, *, nonnegative: bool = False) -> str:\n    number = float(np.float64(value))\n    if not math.isfinite(number):\n        raise ValueError("numeric binding fields must be finite")\n    if nonnegative and number < 0.0:\n        raise ValueError("quoted price must be nonnegative")\n    return number.hex()\n\n\ndef _same_float64(left: Any, right: Any) -> bool:\n    return np.float64(left).tobytes() == np.float64(right).tobytes()\n\n\ndef exact_sql_digest(sql: str) -> str:\n    """Digest exact SQL text; no unsound equivalence normalization is assumed."""\n\n    if not isinstance(sql, str) or not sql.strip():\n        raise ValueError("sql must be a nonempty string")\n    encoded = sql.encode("utf-8")\n    if len(encoded) > MAX_SQL_UTF8_BYTES:\n        raise ValueError(f"sql exceeds {MAX_SQL_UTF8_BYTES} UTF-8 bytes")\n    return _sha256_hex(b"ARBITER-SQL-V1\\x00", encoded)\n\n\ndef _answer_node(value: Any) -> Any:\n    """Convert supported answer values to an unambiguous typed JSON tree."""\n\n    if isinstance(value, np.generic):\n        value = value.item()\n    if value is None:\n        return ["null"]\n    if isinstance(value, bool):\n        return ["bool", value]\n    if isinstance(value, int):\n        return ["int", str(value)]\n    if isinstance(value, float):\n        if math.isnan(value):\n            return ["float", "nan"]\n        if math.isinf(value):\n            return ["float", "+inf" if value > 0 else "-inf"]\n        return ["float", value.hex()]\n    if isinstance(value, Decimal):\n        return ["decimal", str(value)]\n    if isinstance(value, str):\n        return ["str", value]\n    if isinstance(value, bytes):\n        return ["bytes", base64.b64encode(value).decode("ascii")]\n    if isinstance(value, bytearray):\n        return ["bytes", base64.b64encode(bytes(value)).decode("ascii")]\n    if isinstance(value, _datetime.datetime):\n        return ["datetime", value.isoformat()]\n    if isinstance(value, _datetime.date):\n        return ["date", value.isoformat()]\n    if isinstance(value, _datetime.time):\n        return ["time", value.isoformat()]\n    if isinstance(value, (list, tuple)):\n        return ["sequence", [_answer_node(item) for item in value]]\n    if isinstance(value, Mapping):\n        items = []\n        for key, item in value.items():\n            key_node = _answer_node(key)\n            item_node = _answer_node(item)\n            items.append((_canonical_json(key_node), key_node, item_node))\n        items.sort(key=lambda record: record[0])\n        return ["mapping", [[key_node, item_node] for _, key_node, item_node in items]]\n    raise TypeError(f"unsupported answer value type: {type(value).__name__}")\n\n\ndef canonical_answer_bytes(payload: Any) -> bytes:\n    return _canonical_json(["ARBITER-ANSWER-V1", _answer_node(payload)])\n\n\ndef answer_digest(payload_bytes: bytes) -> str:\n    if not isinstance(payload_bytes, bytes):\n        raise TypeError("payload_bytes must be immutable bytes")\n    return _sha256_hex(b"ARBITER-ANSWER-DIGEST-V1\\x00", payload_bytes)\n\n\n@dataclass(frozen=True)\nclass RequestContext:\n    request_id: str\n    buyer_id: str\n    session_id: str\n    asset_id: str\n    query_digest: str\n    quoted_price_hex: str\n    currency: str = "EXPERIMENTAL_PRICE_UNIT"\n\n    def __post_init__(self) -> None:\n        for name in ["request_id", "buyer_id", "session_id", "asset_id", "currency"]:\n            object.__setattr__(self, name, _validated_identifier(name, getattr(self, name)))\n        object.__setattr__(self, "query_digest", _validated_digest("query_digest", self.query_digest))\n        parsed_price = float.fromhex(_validated_identifier("quoted_price_hex", self.quoted_price_hex))\n        if not math.isfinite(parsed_price) or parsed_price < 0.0:\n            raise ValueError("quoted_price_hex must encode a finite nonnegative price")\n        object.__setattr__(self, "quoted_price_hex", parsed_price.hex())\n\n    @classmethod\n    def from_values(\n        cls,\n        *,\n        request_id: str,\n        buyer_id: str,\n        session_id: str,\n        asset_id: str,\n        sql: str,\n        quoted_price: float,\n        currency: str = "EXPERIMENTAL_PRICE_UNIT",\n    ) -> "RequestContext":\n        return cls(\n            request_id=request_id,\n            buyer_id=buyer_id,\n            session_id=session_id,\n            asset_id=asset_id,\n            query_digest=exact_sql_digest(sql),\n            quoted_price_hex=_float64_hex(quoted_price, nonnegative=True),\n            currency=currency,\n        )\n\n    @property\n    def quoted_price(self) -> float:\n        return float.fromhex(self.quoted_price_hex)\n\n    def binding_fields(self) -> dict[str, Any]:\n        return {\n            "request_id": self.request_id,\n            "buyer_id": self.buyer_id,\n            "session_id": self.session_id,\n            "asset_id": self.asset_id,\n            "query_digest": self.query_digest,\n            "quoted_price_hex": self.quoted_price_hex,\n            "currency": self.currency,\n        }\n\n    @property\n    def request_digest(self) -> str:\n        return _sha256_hex(\n            b"ARBITER-REQUEST-V2\\x00",\n            _canonical_json(self.binding_fields()),\n        )\n\n\nclass BoundAnswerBuffer:\n    """Immutable serialized answer plus mutable local release status.\n\n    The payload bytes, buffer identifier, and binding digests are exposed only\n    through read-only properties.  The durable SQLite row remains authoritative\n    after a successful commit, allowing recovery if the process exits before its\n    local ``released`` flag is observed.\n    """\n\n    __slots__ = (\n        "_buffer_id",\n        "_request_digest",\n        "_answer_digest",\n        "_payload_bytes",\n        "_released",\n        "_retained_reason",\n    )\n    _IMMUTABLE_SLOTS = {\n        "_buffer_id",\n        "_request_digest",\n        "_answer_digest",\n        "_payload_bytes",\n    }\n\n    def __setattr__(self, name: str, value: Any) -> None:\n        if name in self._IMMUTABLE_SLOTS and hasattr(self, name):\n            raise AttributeError(f"{name[1:]} is immutable after buffer creation")\n        object.__setattr__(self, name, value)\n\n    def __init__(\n        self,\n        *,\n        request: RequestContext,\n        payload: Any,\n        buffer_id: str | None = None,\n    ) -> None:\n        payload_bytes = canonical_answer_bytes(payload)\n        self._buffer_id = _validated_identifier(\n            "buffer_id", buffer_id if buffer_id is not None else secrets.token_hex(16)\n        )\n        self._request_digest = request.request_digest\n        self._answer_digest = answer_digest(payload_bytes)\n        self._payload_bytes = payload_bytes\n        self._released = False\n        self._retained_reason = "buffered"\n\n    @property\n    def buffer_id(self) -> str:\n        return self._buffer_id\n\n    @property\n    def request_digest(self) -> str:\n        return self._request_digest\n\n    @property\n    def answer_digest(self) -> str:\n        return self._answer_digest\n\n    @property\n    def payload_bytes(self) -> bytes:\n        return self._payload_bytes\n\n    @property\n    def released(self) -> bool:\n        return self._released\n\n    @property\n    def retained_reason(self) -> str:\n        return self._retained_reason\n\n    def _mark_released(self) -> None:\n        object.__setattr__(self, "_released", True)\n        object.__setattr__(self, "_retained_reason", "")\n\n    def retain(self, reason: str) -> None:\n        # Release is monotonic. A replay attempt against the same local object\n        # must not make an already delivered answer appear buffered again.\n        if self._released:\n            return\n        object.__setattr__(self, "_released", False)\n        object.__setattr__(\n            self,\n            "_retained_reason",\n            _validated_identifier("retained_reason", reason),\n        )\n\n\n@dataclass(frozen=True)\nclass RequestBoundReleaseToken:\n    protocol_version: int\n    purpose: str\n    token_id: str\n    nonce: str\n    issued_at_ns: int\n    expires_at_ns: int\n    epoch: int\n    constraint_digest: str\n    asset_digest: str\n    price_digest: str\n    verifier_tolerance_hex: str\n    request_id: str\n    buyer_id: str\n    session_id: str\n    asset_id: str\n    query_digest: str\n    quoted_price_hex: str\n    currency: str\n    request_digest: str\n    buffer_id: str\n    answer_digest: str\n    mac: str\n\n    def signed_fields(self) -> dict[str, Any]:\n        return {\n            "protocol_version": self.protocol_version,\n            "purpose": self.purpose,\n            "token_id": self.token_id,\n            "nonce": self.nonce,\n            "issued_at_ns": self.issued_at_ns,\n            "expires_at_ns": self.expires_at_ns,\n            "epoch": self.epoch,\n            "constraint_digest": self.constraint_digest,\n            "asset_digest": self.asset_digest,\n            "price_digest": self.price_digest,\n            "verifier_tolerance_hex": self.verifier_tolerance_hex,\n            "request_id": self.request_id,\n            "buyer_id": self.buyer_id,\n            "session_id": self.session_id,\n            "asset_id": self.asset_id,\n            "query_digest": self.query_digest,\n            "quoted_price_hex": self.quoted_price_hex,\n            "currency": self.currency,\n            "request_digest": self.request_digest,\n            "buffer_id": self.buffer_id,\n            "answer_digest": self.answer_digest,\n        }\n\n\n@dataclass(frozen=True)\nclass RequestBoundCommitResult:\n    released: bool\n    reason: str\n    epoch: int\n    token_id: str\n    durable: bool\n\n\nclass DurableReleaseLedger:\n    """SQLite-backed staged-answer outbox and release-authorization ledger."""\n\n    def __init__(self, path: str | Path) -> None:\n        self.path = Path(path).resolve()\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self._local = threading.local()\n        self._schema_lock = threading.Lock()\n        self._initialize_schema()\n\n    def _new_connection(self) -> sqlite3.Connection:\n        connection = sqlite3.connect(\n            str(self.path),\n            timeout=30.0,\n            isolation_level=None,\n            check_same_thread=False,\n        )\n        connection.row_factory = sqlite3.Row\n        connection.execute("PRAGMA foreign_keys=ON")\n        connection.execute("PRAGMA busy_timeout=30000")\n        return connection\n\n    def _connection(self) -> sqlite3.Connection:\n        connection = getattr(self._local, "connection", None)\n        if connection is None:\n            connection = self._new_connection()\n            self._local.connection = connection\n        return connection\n\n    def close_thread_connection(self) -> None:\n        connection = getattr(self._local, "connection", None)\n        if connection is not None:\n            connection.close()\n            self._local.connection = None\n\n    def _initialize_schema(self) -> None:\n        with self._schema_lock:\n            connection = self._new_connection()\n            try:\n                connection.execute("PRAGMA journal_mode=WAL")\n                connection.execute("PRAGMA synchronous=FULL")\n                connection.executescript(\n                    """\n                    CREATE TABLE IF NOT EXISTS answer_buffers (\n                        buffer_id TEXT PRIMARY KEY,\n                        request_digest TEXT NOT NULL,\n                        answer_digest TEXT NOT NULL,\n                        payload BLOB NOT NULL,\n                        state TEXT NOT NULL CHECK (state IN (\'buffered\', \'released\')),\n                        staged_at_ns INTEGER NOT NULL,\n                        released_at_ns INTEGER,\n                        token_id TEXT UNIQUE\n                    );\n\n                    CREATE TABLE IF NOT EXISTS release_commits (\n                        token_id TEXT PRIMARY KEY,\n                        request_id TEXT NOT NULL UNIQUE,\n                        buyer_id TEXT NOT NULL,\n                        session_id TEXT NOT NULL,\n                        asset_id TEXT NOT NULL,\n                        query_digest TEXT NOT NULL,\n                        quoted_price_hex TEXT NOT NULL,\n                        currency TEXT NOT NULL,\n                        request_digest TEXT NOT NULL,\n                        buffer_id TEXT NOT NULL UNIQUE,\n                        answer_digest TEXT NOT NULL,\n                        price_digest TEXT NOT NULL,\n                        constraint_digest TEXT NOT NULL,\n                        asset_digest TEXT NOT NULL,\n                        epoch INTEGER NOT NULL,\n                        verifier_tolerance_hex TEXT NOT NULL,\n                        committed_at_ns INTEGER NOT NULL,\n                        FOREIGN KEY(buffer_id) REFERENCES answer_buffers(buffer_id)\n                    );\n\n                    CREATE INDEX IF NOT EXISTS release_commits_buyer_request\n                    ON release_commits(buyer_id, request_id);\n                    """\n                )\n            finally:\n                connection.close()\n\n    def stage_buffer(self, request: RequestContext, buffer: BoundAnswerBuffer) -> None:\n        if type(buffer) is not BoundAnswerBuffer:\n            raise TypeError("buffer must be an exact BoundAnswerBuffer")\n        if buffer.request_digest != request.request_digest:\n            raise ValueError("buffer_request_digest_mismatch")\n        # BoundAnswerBuffer computes this digest in its constructor and exposes\n        # immutable ``bytes`` through read-only slots. Avoid hashing a large\n        # answer twice before it can be staged.\n        connection = self._connection()\n        now_ns = time.time_ns()\n        try:\n            connection.execute("BEGIN IMMEDIATE")\n            existing = connection.execute(\n                """\n                SELECT request_digest, answer_digest, state\n                FROM answer_buffers WHERE buffer_id = ?\n                """,\n                (buffer.buffer_id,),\n            ).fetchone()\n            if existing is None:\n                connection.execute(\n                    """\n                    INSERT INTO answer_buffers(\n                        buffer_id, request_digest, answer_digest, payload,\n                        state, staged_at_ns\n                    ) VALUES (?, ?, ?, ?, \'buffered\', ?)\n                    """,\n                    (\n                        buffer.buffer_id,\n                        buffer.request_digest,\n                        buffer.answer_digest,\n                        sqlite3.Binary(buffer.payload_bytes),\n                        now_ns,\n                    ),\n                )\n            else:\n                identical = (\n                    existing["request_digest"] == buffer.request_digest\n                    and existing["answer_digest"] == buffer.answer_digest\n                    and existing["state"] == "buffered"\n                )\n                if not identical:\n                    raise ValueError("buffer_id_collision_or_already_released")\n            connection.execute("COMMIT")\n        except Exception:\n            if connection.in_transaction:\n                connection.execute("ROLLBACK")\n            raise\n\n    def commit_release(\n        self,\n        token: RequestBoundReleaseToken,\n        request: RequestContext,\n        buffer: BoundAnswerBuffer,\n        *,\n        committed_at_ns: int,\n    ) -> tuple[bool, str]:\n        connection = self._connection()\n        try:\n            connection.execute("BEGIN IMMEDIATE")\n            if connection.execute(\n                "SELECT 1 FROM release_commits WHERE token_id = ?",\n                (token.token_id,),\n            ).fetchone() is not None:\n                connection.execute("ROLLBACK")\n                return False, "token_replay"\n            if connection.execute(\n                "SELECT 1 FROM release_commits WHERE request_id = ?",\n                (request.request_id,),\n            ).fetchone() is not None:\n                connection.execute("ROLLBACK")\n                return False, "request_replay"\n\n            # The answer BLOB was hashed and validated when staged.  Commit reads\n            # only its immutable identity fields, keeping the authorization hot\n            # path independent of answer size. Delivery revalidates the BLOB.\n            staged = connection.execute(\n                """\n                SELECT request_digest, answer_digest, state\n                FROM answer_buffers WHERE buffer_id = ?\n                """,\n                (buffer.buffer_id,),\n            ).fetchone()\n            if staged is None:\n                connection.execute("ROLLBACK")\n                return False, "buffer_not_staged"\n            if staged["state"] != "buffered":\n                connection.execute("ROLLBACK")\n                return False, "buffer_already_released"\n            if staged["request_digest"] != request.request_digest:\n                connection.execute("ROLLBACK")\n                return False, "ledger_request_digest_mismatch"\n            if staged["answer_digest"] != buffer.answer_digest:\n                connection.execute("ROLLBACK")\n                return False, "ledger_answer_digest_mismatch"\n            connection.execute(\n                """\n                INSERT INTO release_commits(\n                    token_id, request_id, buyer_id, session_id, asset_id,\n                    query_digest, quoted_price_hex, currency, request_digest,\n                    buffer_id, answer_digest, price_digest, constraint_digest,\n                    asset_digest, epoch, verifier_tolerance_hex, committed_at_ns\n                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n                """,\n                (\n                    token.token_id,\n                    request.request_id,\n                    request.buyer_id,\n                    request.session_id,\n                    request.asset_id,\n                    request.query_digest,\n                    request.quoted_price_hex,\n                    request.currency,\n                    request.request_digest,\n                    buffer.buffer_id,\n                    buffer.answer_digest,\n                    token.price_digest,\n                    token.constraint_digest,\n                    token.asset_digest,\n                    token.epoch,\n                    token.verifier_tolerance_hex,\n                    int(committed_at_ns),\n                ),\n            )\n            updated = connection.execute(\n                """\n                UPDATE answer_buffers\n                SET state = \'released\', released_at_ns = ?, token_id = ?\n                WHERE buffer_id = ? AND state = \'buffered\'\n                """,\n                (int(committed_at_ns), token.token_id, buffer.buffer_id),\n            ).rowcount\n            if updated != 1:\n                raise RuntimeError("durable buffer state transition was not unique")\n            connection.execute("COMMIT")\n            return True, "released"\n        except sqlite3.IntegrityError as exc:\n            if connection.in_transaction:\n                connection.execute("ROLLBACK")\n            message = str(exc).lower()\n            if "token_id" in message:\n                return False, "token_replay"\n            if "request_id" in message:\n                return False, "request_replay"\n            if "buffer_id" in message:\n                return False, "buffer_already_released"\n            return False, "ledger_integrity_failure"\n        except Exception:\n            if connection.in_transaction:\n                connection.execute("ROLLBACK")\n            raise\n\n    def released_payload(\n        self,\n        *,\n        token_id: str,\n        request: RequestContext,\n    ) -> bytes:\n        token_id = _validated_identifier("token_id", token_id)\n        connection = self._connection()\n        row = connection.execute(\n            """\n            SELECT c.request_digest, c.answer_digest, b.payload, b.state\n            FROM release_commits AS c\n            JOIN answer_buffers AS b ON b.buffer_id = c.buffer_id\n            WHERE c.token_id = ?\n            """,\n            (token_id,),\n        ).fetchone()\n        if row is None or row["state"] != "released":\n            raise KeyError("released answer not found")\n        if row["request_digest"] != request.request_digest:\n            raise PermissionError("delivery_request_mismatch")\n        payload = bytes(row["payload"])\n        if row["answer_digest"] != answer_digest(payload):\n            raise RuntimeError("durable_answer_integrity_failure")\n        return payload\n\n    def release_rows(self) -> list[dict[str, Any]]:\n        connection = self._connection()\n        rows = connection.execute(\n            "SELECT * FROM release_commits ORDER BY committed_at_ns, token_id"\n        ).fetchall()\n        return [dict(row) for row in rows]\n\n    def buffer_rows(self) -> list[dict[str, Any]]:\n        connection = self._connection()\n        rows = connection.execute(\n            """\n            SELECT buffer_id, request_digest, answer_digest, length(payload) AS payload_bytes,\n                   state, staged_at_ns, released_at_ns, token_id\n            FROM answer_buffers ORDER BY staged_at_ns, buffer_id\n            """\n        ).fetchall()\n        return [dict(row) for row in rows]\n\n\nclass RequestBoundSnapshotStore:\n    """Pinned-v4 verifier plus a request-bound, durable compare-and-commit gate."""\n\n    def __init__(\n        self,\n        initial_snapshot: _core.ConstraintSnapshot,\n        *,\n        ledger: DurableReleaseLedger,\n        asset_query_digests: Mapping[str, str],\n        secret: bytes | None = None,\n        verifier_tolerance: float = _core.DEFAULT_TOLERANCE,\n        token_ttl_ns: int = DEFAULT_TOKEN_TTL_NS,\n        clock_ns: Callable[[], int] = time.time_ns,\n    ) -> None:\n        _core._validate_snapshot_identity(initial_snapshot)\n        if not isinstance(ledger, DurableReleaseLedger):\n            raise TypeError("ledger must be a DurableReleaseLedger")\n        self._snapshot = initial_snapshot\n        self._numeric_integrity_digest = _core._numeric_snapshot_integrity_digest(\n            initial_snapshot\n        )\n        self._asset_index = {\n            asset_id: index for index, asset_id in enumerate(initial_snapshot.asset_ids)\n        }\n        supplied_query_digests = {\n            _validated_identifier("asset_id", str(asset_id)): _validated_digest(\n                "query_digest", digest\n            )\n            for asset_id, digest in asset_query_digests.items()\n        }\n        if set(supplied_query_digests) != set(initial_snapshot.asset_ids):\n            raise ValueError(\n                "asset_query_digests must cover exactly the snapshot asset IDs"\n            )\n        self._asset_query_digests = supplied_query_digests\n        self._ledger = ledger\n        self._secret = secret if secret is not None else secrets.token_bytes(32)\n        if not isinstance(self._secret, bytes) or len(self._secret) < 32:\n            raise ValueError("secret must contain at least 32 bytes")\n        self._verifier_tolerance = _validated_tolerance(verifier_tolerance)\n        if isinstance(token_ttl_ns, bool) or int(token_ttl_ns) != token_ttl_ns or int(token_ttl_ns) <= 0:\n            raise ValueError("token_ttl_ns must be a positive integer")\n        self._token_ttl_ns = int(token_ttl_ns)\n        self._clock_ns = clock_ns\n        self._lock = threading.RLock()\n\n    @property\n    def ledger(self) -> DurableReleaseLedger:\n        return self._ledger\n\n    def read(self) -> _core.ConstraintSnapshot:\n        with self._lock:\n            return self._snapshot\n\n    def advance(self, snapshot: _core.ConstraintSnapshot) -> None:\n        with self._lock:\n            if snapshot.epoch <= self._snapshot.epoch:\n                raise ValueError("new snapshot epoch must increase")\n            _core._validate_snapshot_identity(snapshot)\n            if snapshot.asset_ids != self._snapshot.asset_ids:\n                raise ValueError(\n                    "asset registry changes require a new store and query-digest map"\n                )\n            self._snapshot = snapshot\n            self._numeric_integrity_digest = _core._numeric_snapshot_integrity_digest(snapshot)\n            self._asset_index = {\n                asset_id: index for index, asset_id in enumerate(snapshot.asset_ids)\n            }\n\n    def restart_with_new_secret(self) -> None:\n        with self._lock:\n            self._secret = secrets.token_bytes(32)\n\n    def _token_mac(self, signed_fields: Mapping[str, Any]) -> str:\n        return hmac.new(\n            self._secret,\n            _canonical_json(dict(signed_fields)),\n            hashlib.sha256,\n        ).hexdigest()\n\n    def _reject(\n        self,\n        reason: str,\n        token: RequestBoundReleaseToken,\n        buffer: BoundAnswerBuffer,\n    ) -> RequestBoundCommitResult:\n        buffer.retain(reason)\n        return RequestBoundCommitResult(\n            released=False,\n            reason=reason,\n            epoch=self._snapshot.epoch,\n            token_id=getattr(token, "token_id", ""),\n            durable=False,\n        )\n\n    def issue_token(\n        self,\n        snapshot: _core.ConstraintSnapshot,\n        prices: np.ndarray,\n        request: RequestContext,\n        buffer: BoundAnswerBuffer,\n        *,\n        tolerance: float | None = None,\n    ) -> RequestBoundReleaseToken:\n        policy_tolerance = self._verifier_tolerance if tolerance is None else _validated_tolerance(tolerance)\n        if not _same_float64(policy_tolerance, self._verifier_tolerance):\n            raise ValueError("requested tolerance does not match the store policy")\n        if type(request) is not RequestContext:\n            raise TypeError("request must be an exact RequestContext")\n        if type(buffer) is not BoundAnswerBuffer:\n            raise TypeError("buffer must be an exact BoundAnswerBuffer")\n        verification = _core.independent_verify(\n            snapshot,\n            prices,\n            ordered_asset_ids=snapshot.asset_ids,\n            tolerance=policy_tolerance,\n        )\n        if not verification.passed:\n            raise ValueError("cannot issue a release token for an unverified vector")\n        asset_index = {asset_id: index for index, asset_id in enumerate(snapshot.asset_ids)}\n        if request.asset_id not in asset_index:\n            raise ValueError("selected asset is absent from the snapshot")\n        expected_query_digest = self._asset_query_digests.get(request.asset_id)\n        if request.query_digest != expected_query_digest:\n            raise ValueError("query digest does not match the selected asset registry")\n        quoted_from_vector = float(np.asarray(prices, dtype=np.float64)[asset_index[request.asset_id]])\n        if not _same_float64(request.quoted_price, quoted_from_vector):\n            raise ValueError("quoted price does not equal the selected verified vector coordinate")\n        if buffer.request_digest != request.request_digest:\n            raise ValueError("answer buffer is not bound to this request")\n        self._ledger.stage_buffer(request, buffer)\n\n        issued_at_ns = int(self._clock_ns())\n        fields = {\n            "protocol_version": PROTOCOL_VERSION,\n            "purpose": PROTOCOL_PURPOSE,\n            "token_id": secrets.token_hex(16),\n            "nonce": secrets.token_hex(16),\n            "issued_at_ns": issued_at_ns,\n            "expires_at_ns": issued_at_ns + self._token_ttl_ns,\n            "epoch": int(snapshot.epoch),\n            "constraint_digest": snapshot.constraint_digest,\n            "asset_digest": snapshot.asset_digest,\n            "price_digest": _core.price_vector_digest(snapshot.asset_ids, prices),\n            "verifier_tolerance_hex": _float64_hex(policy_tolerance),\n            **request.binding_fields(),\n            "request_digest": request.request_digest,\n            "buffer_id": buffer.buffer_id,\n            "answer_digest": buffer.answer_digest,\n        }\n        return RequestBoundReleaseToken(mac=self._token_mac(fields), **fields)\n\n    def compare_and_commit(\n        self,\n        token: RequestBoundReleaseToken,\n        prices: np.ndarray,\n        request: RequestContext,\n        buffer: BoundAnswerBuffer,\n    ) -> RequestBoundCommitResult:\n        if type(token) is not RequestBoundReleaseToken:\n            raise TypeError("token must be an exact RequestBoundReleaseToken")\n        if type(request) is not RequestContext:\n            raise TypeError("request must be an exact RequestContext")\n        if type(buffer) is not BoundAnswerBuffer:\n            raise TypeError("buffer must be an exact BoundAnswerBuffer")\n\n        with self._lock:\n            current = self._snapshot\n            if _core._numeric_snapshot_integrity_digest(current) != self._numeric_integrity_digest:\n                return self._reject("snapshot_integrity_failure", token, buffer)\n            if token.protocol_version != PROTOCOL_VERSION or token.purpose != PROTOCOL_PURPOSE:\n                return self._reject("protocol_mismatch", token, buffer)\n            try:\n                expected_mac = self._token_mac(token.signed_fields())\n                token_tolerance = float.fromhex(token.verifier_tolerance_hex)\n                token_quote = float.fromhex(token.quoted_price_hex)\n                if not math.isfinite(token_tolerance) or token_tolerance <= 0.0:\n                    raise ValueError("invalid tolerance")\n                if not math.isfinite(token_quote) or token_quote < 0.0:\n                    raise ValueError("invalid quote")\n            except Exception:\n                return self._reject("invalid_token_fields", token, buffer)\n            if not hmac.compare_digest(token.mac, expected_mac):\n                return self._reject("token_mac_mismatch", token, buffer)\n            now_ns = int(self._clock_ns())\n            if now_ns > token.expires_at_ns:\n                return self._reject("token_expired", token, buffer)\n            if token.issued_at_ns > now_ns:\n                return self._reject("token_from_future", token, buffer)\n            if token.epoch != current.epoch:\n                return self._reject("stale_epoch", token, buffer)\n            if token.constraint_digest != current.constraint_digest:\n                return self._reject("constraint_digest_mismatch", token, buffer)\n            if token.asset_digest != current.asset_digest:\n                return self._reject("asset_digest_mismatch", token, buffer)\n            if not _same_float64(token_tolerance, self._verifier_tolerance):\n                return self._reject("tolerance_policy_mismatch", token, buffer)\n\n            expected_request_fields = request.binding_fields()\n            for name, expected in expected_request_fields.items():\n                if getattr(token, name) != expected:\n                    return self._reject(f"{name}_mismatch", token, buffer)\n            if token.request_digest != request.request_digest:\n                return self._reject("request_digest_mismatch", token, buffer)\n            if buffer.request_digest != request.request_digest:\n                return self._reject("buffer_request_mismatch", token, buffer)\n            if token.buffer_id != buffer.buffer_id:\n                return self._reject("buffer_id_mismatch", token, buffer)\n            if token.answer_digest != buffer.answer_digest:\n                return self._reject("answer_digest_mismatch", token, buffer)\n\n            vector = np.asarray(prices, dtype=np.float64)\n            if token.asset_id not in self._asset_index:\n                return self._reject("selected_asset_missing", token, buffer)\n            if token.query_digest != self._asset_query_digests[token.asset_id]:\n                return self._reject("asset_query_registry_mismatch", token, buffer)\n            actual_price_digest = _core.price_vector_digest(current.asset_ids, vector)\n            if token.price_digest != actual_price_digest:\n                return self._reject("price_digest_mismatch", token, buffer)\n            quote_from_vector = float(vector[self._asset_index[token.asset_id]])\n            if not _same_float64(token_quote, quote_from_vector):\n                return self._reject("quote_vector_mismatch", token, buffer)\n            verification = _core.independent_verify(\n                current,\n                vector,\n                ordered_asset_ids=current.asset_ids,\n                tolerance=token_tolerance,\n            )\n            if not verification.passed:\n                return self._reject("commit_verification_failed", token, buffer)\n\n            durable, reason = self._ledger.commit_release(\n                token,\n                request,\n                buffer,\n                committed_at_ns=now_ns,\n            )\n            if not durable:\n                return self._reject(reason, token, buffer)\n            buffer._mark_released()\n            return RequestBoundCommitResult(\n                released=True,\n                reason="released",\n                epoch=current.epoch,\n                token_id=token.token_id,\n                durable=True,\n            )\n\n\ndef replace_bound_token(\n    token: RequestBoundReleaseToken,\n    **changes: Any,\n) -> RequestBoundReleaseToken:\n    """Fault-injection helper; changed signed fields invalidate the MAC."""\n\n    return replace(token, **changes)\n\n\ndef next_epoch_snapshot(snapshot: _core.ConstraintSnapshot) -> _core.ConstraintSnapshot:\n    """Construct an identity-equivalent snapshot at the next epoch for tests."""\n\n    return _core.ConstraintSnapshot(\n        asset_ids=snapshot.asset_ids,\n        matrix=snapshot.matrix,\n        bounds=snapshot.bounds,\n        row_metadata=snapshot.row_metadata,\n        epoch=snapshot.epoch + 1,\n        constraint_digest=snapshot.constraint_digest,\n        asset_digest=snapshot.asset_digest,\n    )\n',
    "arbiter_closure_experiments.py": '"""Targeted experiments for the ARBITER request-binding closure campaign."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, replace\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport secrets\nimport sys\nimport tempfile\nimport threading\nimport time\nfrom typing import Any, Callable, Iterable, Mapping, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nimport arbiter_core as core\nfrom arbiter_experiments import relation_family_attack_center\nfrom arbiter_request_release import (\n    BoundAnswerBuffer,\n    DurableReleaseLedger,\n    RequestBoundSnapshotStore,\n    RequestContext,\n    exact_sql_digest,\n    next_epoch_snapshot,\n    replace_bound_token,\n)\n\n\nAUDITED_SUITE_VERSION = "2026-08-01-audited-v4"\nCLOSURE_SUITE_VERSION = "2026-08-01-request-bound-closure-v1.1"\nAUDITED_CODE_HASHES = {\n    "arbiter_core.py": "b6797381dd883eac5d3cef7f1f86eaa2ad66bf73c498fdcfb7493f5113175ffa",\n    "arbiter_data.py": "fb4d85f6a36be12bf1dc3ac09e80383f4d7e52e83a923bae1840a87443452c86",\n    "arbiter_experiments.py": "f5919e83b4c2658950a611e94f3bc8de62000a4ab7bd1cb24b90f64e5a0b590e",\n    "arbiter_workload.py": "cd3a78b28c573698e77fe02bc301273bbc959d770ee6daadae52ace68151ef5a",\n}\nRELATION_FAMILIES = (\n    "decomposition",\n    "projection_derivation",\n    "filter_derivation",\n)\n\n\ndef sha256_file(path: Path, chunk_bytes: int = 1024 * 1024) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        while True:\n            chunk = handle.read(chunk_bytes)\n            if not chunk:\n                break\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_audited_inputs(\n    run_dir: Path,\n    code_dir: Path,\n    *,\n    active_run_dir: Path | None = None,\n) -> pd.DataFrame:\n    """Fail closed on stale, partial, modified, or non-full audited inputs.\n\n    ``run_dir`` is the immutable evidence-package root created by Notebook 04.\n    When ``active_run_dir`` is supplied, every packaged stage file is also\n    checked against the live 00--04 run.  This prevents a stale nested package\n    from being accepted merely because it is internally self-consistent.\n    """\n\n    run_dir = run_dir.resolve()\n    code_dir = code_dir.resolve()\n    config_path = run_dir / "run_config.json"\n    checksums_path = run_dir / "checksums.csv"\n    quality_path = run_dir / "04_evidence" / "all_stage_quality_checks.csv"\n    for required in [config_path, checksums_path, quality_path]:\n        if not required.is_file():\n            raise FileNotFoundError(f"required audited input is missing: {required}")\n\n    config = json.loads(config_path.read_text(encoding="utf-8"))\n    if config.get("run_mode") != "full":\n        raise RuntimeError("closure evidence requires a completed full run")\n    if config.get("suite_version") != AUDITED_SUITE_VERSION:\n        raise RuntimeError("closure input is not the audited-v4 suite")\n\n    quality = pd.read_csv(quality_path)\n    passed = quality["passed"].astype(str).str.lower().map({"true": True, "false": False})\n    if passed.isna().any() or not passed.all() or len(quality) != 52:\n        raise RuntimeError("the prerequisite 52 audited-v4 stage checks do not all pass")\n\n    rows: list[dict[str, Any]] = []\n    manifest = pd.read_csv(checksums_path)\n    required_columns = {"path", "bytes", "sha256"}\n    if not required_columns.issubset(manifest.columns):\n        raise RuntimeError("audited checksum manifest has an unexpected schema")\n    if manifest.empty:\n        raise RuntimeError("audited checksum manifest is empty")\n    if manifest["path"].isna().any() or manifest["path"].astype(str).duplicated().any():\n        raise RuntimeError("audited checksum manifest has missing or duplicate paths")\n    if manifest["bytes"].isna().any():\n        raise RuntimeError("audited checksum manifest has missing byte counts")\n    byte_counts = pd.to_numeric(manifest["bytes"], errors="coerce")\n    if byte_counts.isna().any() or (byte_counts < 0).any() or not np.all(\n        byte_counts.to_numpy(float) == np.floor(byte_counts.to_numpy(float))\n    ):\n        raise RuntimeError("audited checksum manifest has invalid byte counts")\n    digests = manifest["sha256"].astype(str)\n    if not digests.str.fullmatch(r"[0-9a-f]{64}").all():\n        raise RuntimeError("audited checksum manifest has invalid SHA-256 values")\n\n    sha_text_path = run_dir / "checksums.sha256"\n    if not sha_text_path.is_file():\n        raise FileNotFoundError(\n            f"required audited input is missing: {sha_text_path}"\n        )\n    expected_sha_text = "".join(\n        f"{row.sha256}  {row.path}\\n" for row in manifest.itertuples(index=False)\n    )\n    if sha_text_path.read_text(encoding="utf-8") != expected_sha_text:\n        raise RuntimeError("audited CSV and SHA-256 manifests disagree")\n\n    for record in manifest.itertuples(index=False):\n        relative = Path(str(record.path))\n        if relative.is_absolute() or ".." in relative.parts:\n            raise RuntimeError(f"unsafe checksum path: {relative}")\n        target = (run_dir / relative).resolve()\n        if run_dir not in target.parents:\n            raise RuntimeError(f"checksum path escaped run directory: {relative}")\n        exists = target.is_file()\n        observed_bytes = target.stat().st_size if exists else -1\n        observed_hash = sha256_file(target) if exists else ""\n        row_passed = bool(\n            exists\n            and observed_bytes == int(record.bytes)\n            and observed_hash == str(record.sha256)\n        )\n        rows.append(\n            {\n                "scope": "run_evidence",\n                "path": relative.as_posix(),\n                "expected_bytes": int(record.bytes),\n                "observed_bytes": int(observed_bytes),\n                "expected_sha256": str(record.sha256),\n                "observed_sha256": observed_hash,\n                "passed": row_passed,\n            }\n        )\n\n    if active_run_dir is not None:\n        active_run_dir = active_run_dir.resolve()\n        if not active_run_dir.is_dir():\n            raise FileNotFoundError(\n                f"active audited run directory is missing: {active_run_dir}"\n            )\n        if active_run_dir != run_dir:\n            live_roots = {\n                "00_workload",\n                "01_proposals_attacks",\n                "02_release_protocol",\n                "03_scalability",\n                "04_evidence",\n            }\n            linked_records = 0\n            for record in manifest.itertuples(index=False):\n                relative = Path(str(record.path))\n                is_live_output = (\n                    relative.as_posix() == "run_config.json"\n                    or (relative.parts and relative.parts[0] in live_roots)\n                )\n                if not is_live_output:\n                    continue\n                linked_records += 1\n                target = (active_run_dir / relative).resolve()\n                if active_run_dir not in target.parents:\n                    raise RuntimeError(\n                        f"active-run checksum path escaped its directory: {relative}"\n                    )\n                exists = target.is_file()\n                observed_bytes = target.stat().st_size if exists else -1\n                observed_hash = sha256_file(target) if exists else ""\n                row_passed = bool(\n                    exists\n                    and observed_bytes == int(record.bytes)\n                    and observed_hash == str(record.sha256)\n                )\n                rows.append(\n                    {\n                        "scope": "active_run_package_link",\n                        "path": relative.as_posix(),\n                        "expected_bytes": int(record.bytes),\n                        "observed_bytes": int(observed_bytes),\n                        "expected_sha256": str(record.sha256),\n                        "observed_sha256": observed_hash,\n                        "passed": row_passed,\n                    }\n                )\n            if linked_records == 0:\n                raise RuntimeError(\n                    "audited manifest contains no live 00-04 outputs to link"\n                )\n\n    for name, expected_hash in AUDITED_CODE_HASHES.items():\n        target = code_dir / name\n        observed_hash = sha256_file(target) if target.is_file() else ""\n        rows.append(\n            {\n                "scope": "pinned_code",\n                "path": name,\n                "expected_bytes": None,\n                "observed_bytes": target.stat().st_size if target.is_file() else -1,\n                "expected_sha256": expected_hash,\n                "observed_sha256": observed_hash,\n                "passed": observed_hash == expected_hash,\n            }\n        )\n    audit = pd.DataFrame(rows)\n    if not audit["passed"].all():\n        failed = audit.loc[~audit["passed"], "path"].tolist()\n        raise RuntimeError(f"audited input integrity failed: {failed[:5]}")\n    return audit\n\n\n@dataclass\nclass DatasetBundle:\n    dataset: str\n    assets: pd.DataFrame\n    relations: pd.DataFrame\n    snapshot: core.ConstraintSnapshot\n    prices: np.ndarray\n    query_digests: dict[str, str]\n\n\n@dataclass\nclass TieredRepairResult:\n    prices: np.ndarray\n    verification: core.VerificationResult\n    rejected: bool\n    fallback_used: bool\n    fallback_stage: str\n    solver: str\n    sweeps: int\n    projected_rows: int\n    runtime_ms: float\n    fallback_ms: float\n    l1_adjustment: float\n    error: str\n\n\ndef run_tiered_fail_safe_repair(\n    snapshot: core.ConstraintSnapshot,\n    candidate: np.ndarray,\n    *,\n    tolerance: float,\n    max_sweeps: int,\n    active_row_budget: int,\n    step: float,\n    osqp_max_iter: int,\n    full_fallback_sweeps: int = 20,\n) -> TieredRepairResult:\n    """Use the cheapest verified repair tier before invoking a QP solver.\n\n    Dense cases showed that constructing and iterating OSQP can dominate a\n    fallback even when two full cyclic sweeps suffice. Every tier is followed by\n    the independent verifier; a failed final tier is rejected.\n    """\n\n    original = np.asarray(candidate, dtype=np.float64)\n    active = core.run_enforcement_method(\n        snapshot,\n        original,\n        method="arbiter_active",\n        tolerance=tolerance,\n        max_sweeps=max_sweeps,\n        active_row_budget=active_row_budget,\n        step=step,\n        allow_fallback=False,\n        osqp_max_iter=osqp_max_iter,\n    )\n    if active.verification.passed:\n        return TieredRepairResult(\n            prices=active.prices,\n            verification=active.verification,\n            rejected=False,\n            fallback_used=False,\n            fallback_stage="none",\n            solver=active.solver,\n            sweeps=active.sweeps,\n            projected_rows=active.projected_rows,\n            runtime_ms=active.runtime_ms,\n            fallback_ms=0.0,\n            l1_adjustment=float(np.abs(active.prices - original).sum()),\n            error=active.error,\n        )\n\n    full = core.run_enforcement_method(\n        snapshot,\n        active.prices,\n        method="full_cyclic",\n        tolerance=tolerance,\n        max_sweeps=full_fallback_sweeps,\n        active_row_budget=snapshot.constraints,\n        step=step,\n        allow_fallback=False,\n        osqp_max_iter=osqp_max_iter,\n    )\n    total_runtime = active.runtime_ms + full.runtime_ms\n    total_sweeps = active.sweeps + full.sweeps\n    total_projected = active.projected_rows + full.projected_rows\n    if full.verification.passed:\n        return TieredRepairResult(\n            prices=full.prices,\n            verification=full.verification,\n            rejected=False,\n            fallback_used=True,\n            fallback_stage="full_cyclic",\n            solver="active_cyclic+full_cyclic",\n            sweeps=total_sweeps,\n            projected_rows=total_projected,\n            runtime_ms=total_runtime,\n            fallback_ms=full.runtime_ms,\n            l1_adjustment=float(np.abs(full.prices - original).sum()),\n            error="; ".join(part for part in [active.error, full.error] if part),\n        )\n\n    qp = core.run_enforcement_method(\n        snapshot,\n        original,\n        method="osqp_qp",\n        tolerance=tolerance,\n        max_sweeps=max_sweeps,\n        active_row_budget=active_row_budget,\n        step=step,\n        allow_fallback=False,\n        osqp_max_iter=osqp_max_iter,\n    )\n    total_runtime += qp.runtime_ms\n    final_verification = core.independent_verify(\n        snapshot, qp.prices, tolerance=tolerance\n    )\n    rejected = not final_verification.passed\n    return TieredRepairResult(\n        prices=qp.prices,\n        verification=final_verification,\n        rejected=rejected,\n        fallback_used=True,\n        fallback_stage="osqp" if not rejected else "rejected_after_osqp",\n        solver="active_cyclic+full_cyclic+osqp",\n        sweeps=total_sweeps + qp.sweeps,\n        projected_rows=total_projected + qp.projected_rows,\n        runtime_ms=total_runtime,\n        fallback_ms=full.runtime_ms + qp.runtime_ms,\n        l1_adjustment=float(np.abs(qp.prices - original).sum()),\n        error="; ".join(\n            part for part in [active.error, full.error, qp.error] if part\n        ),\n    )\n\n\ndef build_dataset_bundles(\n    assets: pd.DataFrame,\n    relations: pd.DataFrame,\n    policy: pd.DataFrame,\n    *,\n    tolerance: float,\n) -> dict[str, DatasetBundle]:\n    supported = assets[\n        assets["expected_supported"].astype(str).str.lower().eq("true")\n    ].copy()\n    if supported.duplicated(["dataset", "asset_id"]).any():\n        raise ValueError("supported asset identifiers are not unique per dataset")\n\n    bundles: dict[str, DatasetBundle] = {}\n    for dataset, dataset_assets in supported.groupby("dataset", sort=False):\n        dataset_assets = dataset_assets.reset_index(drop=True)\n        asset_ids = dataset_assets["asset_id"].astype(str).tolist()\n        dataset_relations = relations[relations["dataset"] == dataset].copy()\n        snapshot = core.build_snapshot(asset_ids, dataset_relations, epoch=1)\n        price_rows = policy[policy["dataset"] == dataset][\n            ["asset_id", "target_price"]\n        ]\n        if price_rows["asset_id"].duplicated().any():\n            raise ValueError(f"duplicate controlled prices for {dataset}")\n        price_map = price_rows.set_index("asset_id")["target_price"].to_dict()\n        if set(price_map) != set(asset_ids):\n            raise ValueError(f"controlled price coverage mismatch for {dataset}")\n        prices = np.asarray([price_map[asset_id] for asset_id in asset_ids], dtype=np.float64)\n        verification = core.independent_verify(snapshot, prices, tolerance=tolerance)\n        if not verification.passed:\n            raise RuntimeError(f"controlled policy is infeasible for {dataset}")\n        query_digests = {\n            str(row.asset_id): exact_sql_digest(str(row.sql))\n            for row in dataset_assets.itertuples(index=False)\n        }\n        bundles[str(dataset)] = DatasetBundle(\n            dataset=str(dataset),\n            assets=dataset_assets,\n            relations=dataset_relations,\n            snapshot=snapshot,\n            prices=prices,\n            query_digests=query_digests,\n        )\n    if len(bundles) != 3:\n        raise RuntimeError(f"expected three audited datasets, found {len(bundles)}")\n    return bundles\n\n\ndef _safe_slug(value: str) -> str:\n    slug = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")\n    return slug[:80] or "dataset"\n\n\ndef _protocol_assets(bundle: DatasetBundle) -> tuple[pd.Series, pd.Series]:\n    scalar = bundle.assets[\n        bundle.assets["family"].astype(str).str.contains("aggregate_partition", na=False)\n    ].sort_values("asset_id")\n    if len(scalar) < 2:\n        raise RuntimeError(f"{bundle.dataset} lacks two scalar protocol assets")\n    return scalar.iloc[0], scalar.iloc[1]\n\n\ndef _request_for_row(\n    row: pd.Series,\n    prices_by_asset: Mapping[str, float],\n    *,\n    request_id: str,\n    buyer_id: str = "buyer-a",\n    session_id: str = "session-a",\n) -> RequestContext:\n    asset_id = str(row["asset_id"])\n    return RequestContext.from_values(\n        request_id=request_id,\n        buyer_id=buyer_id,\n        session_id=session_id,\n        asset_id=asset_id,\n        sql=str(row["sql"]),\n        quoted_price=float(prices_by_asset[asset_id]),\n    )\n\n\nclass _MutableClock:\n    def __init__(self, value: int) -> None:\n        self.value = int(value)\n\n    def __call__(self) -> int:\n        return self.value\n\n\ndef run_protocol_fault_campaign(\n    bundles: Mapping[str, DatasetBundle],\n    *,\n    answer_executor: Callable[[str, str], Any],\n    work_dir: Path,\n    tolerance: float,\n    concurrency_trials: int,\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Exercise field substitutions, replay, mutation, expiry, and races."""\n\n    work_dir.mkdir(parents=True, exist_ok=True)\n    fault_rows: list[dict[str, Any]] = []\n    concurrency_rows: list[dict[str, Any]] = []\n    delivery_rows: list[dict[str, Any]] = []\n    durable_rows: list[dict[str, Any]] = []\n\n    def record(\n        *,\n        dataset: str,\n        case: str,\n        expected_release: bool,\n        result,\n        buffer: BoundAnswerBuffer | None,\n        ledger: DurableReleaseLedger,\n        expected_reasons: Iterable[str] = (),\n        phase: str = "commit",\n    ) -> None:\n        release_count = len(ledger.release_rows())\n        observed_release = bool(result.released) if result is not None else False\n        reason = str(result.reason) if result is not None else "issuance_rejected"\n        reason_ok = not expected_reasons or reason in set(expected_reasons)\n        passed = bool(\n            observed_release == expected_release\n            and reason_ok\n            and release_count == (1 if expected_release else 0)\n            and (\n                buffer is None\n                or bool(buffer.released) == expected_release\n                or (case == "token_replay" and buffer.released)\n            )\n        )\n        fault_rows.append(\n            {\n                "dataset": dataset,\n                "case": case,\n                "phase": phase,\n                "expected_release": expected_release,\n                "released": observed_release,\n                "durable": bool(result.durable) if result is not None else False,\n                "reason": reason,\n                "buffer_released": bool(buffer.released) if buffer is not None else False,\n                "durable_release_rows": release_count,\n                "passed": passed,\n            }\n        )\n\n    for dataset, bundle in bundles.items():\n        dataset_dir = work_dir / _safe_slug(dataset)\n        dataset_dir.mkdir(parents=True, exist_ok=True)\n        row_a, row_b = _protocol_assets(bundle)\n        price_map = dict(zip(bundle.snapshot.asset_ids, bundle.prices, strict=True))\n        answer_a = answer_executor(dataset, str(row_a["sql"]))\n        answer_b = answer_executor(dataset, str(row_b["sql"]))\n        if answer_a == answer_b:\n            # Distinct query/asset bindings remain meaningful even if two scalar\n            # aggregate values happen to coincide. The answer-swap case below\n            # uses a typed wrapper so its digest is certainly distinct.\n            answer_b = {"asset_id": str(row_b["asset_id"]), "rows": answer_b}\n\n        case_counter = 0\n\n        def prepared(\n            label: str,\n            *,\n            clock: Callable[[], int] | None = None,\n            ttl_ns: int = 10_000_000_000,\n            secret: bytes | None = None,\n        ):\n            nonlocal case_counter\n            case_counter += 1\n            ledger_path = dataset_dir / f"{case_counter:03d}-{_safe_slug(label)}.sqlite"\n            if ledger_path.exists():\n                ledger_path.unlink()\n            ledger = DurableReleaseLedger(ledger_path)\n            service_secret = secret if secret is not None else secrets.token_bytes(32)\n            store = RequestBoundSnapshotStore(\n                bundle.snapshot,\n                ledger=ledger,\n                asset_query_digests=bundle.query_digests,\n                secret=service_secret,\n                verifier_tolerance=tolerance,\n                token_ttl_ns=ttl_ns,\n                clock_ns=clock if clock is not None else time.time_ns,\n            )\n            request = _request_for_row(\n                row_a,\n                price_map,\n                request_id=f"{_safe_slug(dataset)}-{label}-{case_counter}",\n            )\n            buffer = BoundAnswerBuffer(request=request, payload=answer_a)\n            token = store.issue_token(bundle.snapshot, bundle.prices, request, buffer)\n            return ledger, store, request, buffer, token, service_secret\n\n        # Positive control.\n        ledger, store, request, buffer, token, _ = prepared("normal")\n        result = store.compare_and_commit(token, bundle.prices, request, buffer)\n        record(\n            dataset=dataset,\n            case="normal_request_bound_release",\n            expected_release=True,\n            result=result,\n            buffer=buffer,\n            ledger=ledger,\n            expected_reasons={"released"},\n        )\n        recovered = ledger.released_payload(token_id=token.token_id, request=request)\n        delivery_rows.append(\n            {\n                "dataset": dataset,\n                "case": "authorized_delivery",\n                "delivered": recovered == buffer.payload_bytes,\n                "reason": "delivered" if recovered == buffer.payload_bytes else "payload_mismatch",\n                "passed": recovered == buffer.payload_bytes,\n            }\n        )\n        wrong_delivery_request = replace(request, buyer_id="buyer-substituted")\n        try:\n            ledger.released_payload(token_id=token.token_id, request=wrong_delivery_request)\n            wrong_delivery_passed = False\n            wrong_delivery_reason = "unexpected_delivery"\n        except PermissionError as exc:\n            wrong_delivery_passed = True\n            wrong_delivery_reason = str(exc)\n        delivery_rows.append(\n            {\n                "dataset": dataset,\n                "case": "cross_buyer_delivery",\n                "delivered": False,\n                "reason": wrong_delivery_reason,\n                "passed": wrong_delivery_passed,\n            }\n        )\n        durable_rows.extend({"dataset": dataset, **row} for row in ledger.release_rows())\n\n        substitution_builders = {\n            "request_id_swap": lambda original: replace(original, request_id=original.request_id + "-swap"),\n            "buyer_id_swap": lambda original: replace(original, buyer_id="buyer-b"),\n            "session_id_swap": lambda original: replace(original, session_id="session-b"),\n            "query_swap": lambda original: replace(\n                original, query_digest=exact_sql_digest(str(row_b["sql"]))\n            ),\n            "quoted_price_swap": lambda original: replace(\n                original,\n                quoted_price_hex=float(np.nextafter(original.quoted_price, math.inf)).hex(),\n            ),\n        }\n        for label, builder in substitution_builders.items():\n            ledger, store, request, buffer, token, _ = prepared(label)\n            substituted_request = builder(request)\n            substituted_buffer = BoundAnswerBuffer(\n                request=substituted_request,\n                payload=answer_a,\n            )\n            result = store.compare_and_commit(\n                token, bundle.prices, substituted_request, substituted_buffer\n            )\n            record(\n                dataset=dataset,\n                case=label,\n                expected_release=False,\n                result=result,\n                buffer=substituted_buffer,\n                ledger=ledger,\n            )\n\n        # Cross-asset request contains the registered SQL and quote for asset B.\n        ledger, store, request, buffer, token, _ = prepared("asset-swap")\n        request_b = _request_for_row(\n            row_b,\n            price_map,\n            request_id=request.request_id,\n            buyer_id=request.buyer_id,\n            session_id=request.session_id,\n        )\n        buffer_b = BoundAnswerBuffer(request=request_b, payload=answer_b)\n        result = store.compare_and_commit(token, bundle.prices, request_b, buffer_b)\n        record(\n            dataset=dataset,\n            case="asset_swap",\n            expected_release=False,\n            result=result,\n            buffer=buffer_b,\n            ledger=ledger,\n        )\n\n        # Same request metadata but a different answer/buffer.\n        ledger, store, request, buffer, token, _ = prepared("answer-swap")\n        swapped_buffer = BoundAnswerBuffer(\n            request=request,\n            payload={"substituted": answer_b},\n        )\n        result = store.compare_and_commit(token, bundle.prices, request, swapped_buffer)\n        record(\n            dataset=dataset,\n            case="answer_buffer_swap",\n            expected_release=False,\n            result=result,\n            buffer=swapped_buffer,\n            ledger=ledger,\n            expected_reasons={"buffer_id_mismatch"},\n        )\n\n        # Valid token A used against complete request/buffer B.\n        ledger, store, request, buffer, token, _ = prepared("cross-request-token")\n        request_b = _request_for_row(\n            row_b,\n            price_map,\n            request_id=request.request_id + "-b",\n            buyer_id="buyer-b",\n            session_id="session-b",\n        )\n        buffer_b = BoundAnswerBuffer(request=request_b, payload=answer_b)\n        result = store.compare_and_commit(token, bundle.prices, request_b, buffer_b)\n        record(\n            dataset=dataset,\n            case="cross_request_token_swap",\n            expected_release=False,\n            result=result,\n            buffer=buffer_b,\n            ledger=ledger,\n        )\n\n        ledger, store, request, buffer, token, _ = prepared("price-vector-swap")\n        changed_prices = bundle.prices.copy()\n        changed_prices[1] = np.nextafter(changed_prices[1], math.inf)\n        result = store.compare_and_commit(token, changed_prices, request, buffer)\n        record(\n            dataset=dataset,\n            case="price_vector_swap",\n            expected_release=False,\n            result=result,\n            buffer=buffer,\n            ledger=ledger,\n            expected_reasons={"price_digest_mismatch"},\n        )\n\n        ledger, store, request, buffer, token, _ = prepared("signed-field-tamper")\n        tampered = replace_bound_token(token, buyer_id="tampered-buyer")\n        result = store.compare_and_commit(tampered, bundle.prices, request, buffer)\n        record(\n            dataset=dataset,\n            case="signed_field_tamper",\n            expected_release=False,\n            result=result,\n            buffer=buffer,\n            ledger=ledger,\n            expected_reasons={"token_mac_mismatch"},\n        )\n\n        ledger, store, request, buffer, token, _ = prepared("mac-tamper")\n        tampered = replace_bound_token(token, mac="0" * 64)\n        result = store.compare_and_commit(tampered, bundle.prices, request, buffer)\n        record(\n            dataset=dataset,\n            case="mac_tamper",\n            expected_release=False,\n            result=result,\n            buffer=buffer,\n            ledger=ledger,\n            expected_reasons={"token_mac_mismatch"},\n        )\n\n        ledger, store, request, buffer, token, _ = prepared("stale-epoch")\n        store.advance(next_epoch_snapshot(bundle.snapshot))\n        result = store.compare_and_commit(token, bundle.prices, request, buffer)\n        record(\n            dataset=dataset,\n            case="stale_epoch",\n            expected_release=False,\n            result=result,\n            buffer=buffer,\n            ledger=ledger,\n            expected_reasons={"stale_epoch"},\n        )\n\n        clock = _MutableClock(1_000)\n        ledger, store, request, buffer, token, _ = prepared(\n            "expired-token", clock=clock, ttl_ns=10\n        )\n        clock.value = 1_011\n        result = store.compare_and_commit(token, bundle.prices, request, buffer)\n        record(\n            dataset=dataset,\n            case="expired_token",\n            expected_release=False,\n            result=result,\n            buffer=buffer,\n            ledger=ledger,\n            expected_reasons={"token_expired"},\n        )\n\n        ledger, store, request, buffer, token, _ = prepared("secret-rotation")\n        store.restart_with_new_secret()\n        result = store.compare_and_commit(token, bundle.prices, request, buffer)\n        record(\n            dataset=dataset,\n            case="secret_rotation",\n            expected_release=False,\n            result=result,\n            buffer=buffer,\n            ledger=ledger,\n            expected_reasons={"token_mac_mismatch"},\n        )\n\n        # Safety-critical sparse arrays are rehashed on the commit path.\n        mutable_matrix = bundle.snapshot.matrix.copy()\n        mutable_bounds = bundle.snapshot.bounds.copy()\n        mutable_snapshot = core.ConstraintSnapshot(\n            asset_ids=bundle.snapshot.asset_ids,\n            matrix=mutable_matrix,\n            bounds=mutable_bounds,\n            row_metadata=bundle.snapshot.row_metadata.copy(deep=True),\n            epoch=bundle.snapshot.epoch,\n            constraint_digest=bundle.snapshot.constraint_digest,\n            asset_digest=bundle.snapshot.asset_digest,\n        )\n        mutation_path = dataset_dir / "snapshot-mutation.sqlite"\n        if mutation_path.exists():\n            mutation_path.unlink()\n        ledger = DurableReleaseLedger(mutation_path)\n        store = RequestBoundSnapshotStore(\n            mutable_snapshot,\n            ledger=ledger,\n            asset_query_digests=bundle.query_digests,\n            secret=secrets.token_bytes(32),\n            verifier_tolerance=tolerance,\n        )\n        request = _request_for_row(\n            row_a,\n            price_map,\n            request_id=f"{_safe_slug(dataset)}-snapshot-mutation",\n        )\n        buffer = BoundAnswerBuffer(request=request, payload=answer_a)\n        token = store.issue_token(\n            mutable_snapshot, bundle.prices, request, buffer\n        )\n        mutable_matrix.data[0] += 0.25\n        result = store.compare_and_commit(\n            token, bundle.prices, request, buffer\n        )\n        record(\n            dataset=dataset,\n            case="snapshot_mutation_after_issue",\n            expected_release=False,\n            result=result,\n            buffer=buffer,\n            ledger=ledger,\n            expected_reasons={"snapshot_integrity_failure"},\n        )\n\n        ledger, store, request, buffer, token, _ = prepared("token-replay")\n        first = store.compare_and_commit(token, bundle.prices, request, buffer)\n        replay = store.compare_and_commit(token, bundle.prices, request, buffer)\n        fault_rows.append(\n            {\n                "dataset": dataset,\n                "case": "token_replay",\n                "phase": "second_commit",\n                "expected_release": False,\n                "released": replay.released,\n                "durable": replay.durable,\n                "reason": replay.reason,\n                "buffer_released": buffer.released,\n                "durable_release_rows": len(ledger.release_rows()),\n                "passed": bool(\n                    first.released\n                    and not replay.released\n                    and replay.reason == "token_replay"\n                    and len(ledger.release_rows()) == 1\n                ),\n            }\n        )\n\n        # A request identifier is an idempotency key even when two independently\n        # valid tokens and buffers exist for it.\n        ledger_path = dataset_dir / "request-replay.sqlite"\n        if ledger_path.exists():\n            ledger_path.unlink()\n        ledger = DurableReleaseLedger(ledger_path)\n        shared_secret = secrets.token_bytes(32)\n        store = RequestBoundSnapshotStore(\n            bundle.snapshot,\n            ledger=ledger,\n            asset_query_digests=bundle.query_digests,\n            secret=shared_secret,\n            verifier_tolerance=tolerance,\n        )\n        request = _request_for_row(\n            row_a,\n            price_map,\n            request_id=f"{_safe_slug(dataset)}-request-replay",\n        )\n        first_buffer = BoundAnswerBuffer(request=request, payload=answer_a)\n        second_buffer = BoundAnswerBuffer(request=request, payload=answer_a)\n        first_token = store.issue_token(\n            bundle.snapshot, bundle.prices, request, first_buffer\n        )\n        second_token = store.issue_token(\n            bundle.snapshot, bundle.prices, request, second_buffer\n        )\n        first_result = store.compare_and_commit(\n            first_token, bundle.prices, request, first_buffer\n        )\n        second_result = store.compare_and_commit(\n            second_token, bundle.prices, request, second_buffer\n        )\n        fault_rows.append(\n            {\n                "dataset": dataset,\n                "case": "request_replay_distinct_token",\n                "phase": "second_commit",\n                "expected_release": False,\n                "released": second_result.released,\n                "durable": second_result.durable,\n                "reason": second_result.reason,\n                "buffer_released": second_buffer.released,\n                "durable_release_rows": len(ledger.release_rows()),\n                "passed": bool(\n                    first_result.released\n                    and not second_result.released\n                    and second_result.reason == "request_replay"\n                    and len(ledger.release_rows()) == 1\n                ),\n            }\n        )\n\n        ledger, store, request, buffer, token, shared_secret = prepared(\n            "durable-replay"\n        )\n        first = store.compare_and_commit(token, bundle.prices, request, buffer)\n        ledger.close_thread_connection()\n        reopened = DurableReleaseLedger(ledger.path)\n        restarted_store = RequestBoundSnapshotStore(\n            bundle.snapshot,\n            ledger=reopened,\n            asset_query_digests=bundle.query_digests,\n            secret=shared_secret,\n            verifier_tolerance=tolerance,\n        )\n        replay_buffer = BoundAnswerBuffer(\n            request=request,\n            payload=answer_a,\n            buffer_id=buffer.buffer_id,\n        )\n        replay = restarted_store.compare_and_commit(\n            token, bundle.prices, request, replay_buffer\n        )\n        fault_rows.append(\n            {\n                "dataset": dataset,\n                "case": "durable_replay_after_restart",\n                "phase": "restart_commit",\n                "expected_release": False,\n                "released": replay.released,\n                "durable": replay.durable,\n                "reason": replay.reason,\n                "buffer_released": replay_buffer.released,\n                "durable_release_rows": len(reopened.release_rows()),\n                "passed": bool(\n                    first.released\n                    and not replay.released\n                    and replay.reason == "token_replay"\n                    and len(reopened.release_rows()) == 1\n                ),\n            }\n        )\n\n        # Invalid registry and quote combinations must fail before token issuance.\n        for label, invalid_request in [\n            (\n                "unregistered_query_at_issue",\n                replace(\n                    _request_for_row(\n                        row_a,\n                        price_map,\n                        request_id=f"{_safe_slug(dataset)}-invalid-query",\n                    ),\n                    query_digest=exact_sql_digest("SELECT 999 AS invalid"),\n                ),\n            ),\n            (\n                "wrong_quote_at_issue",\n                replace(\n                    _request_for_row(\n                        row_a,\n                        price_map,\n                        request_id=f"{_safe_slug(dataset)}-invalid-quote",\n                    ),\n                    quoted_price_hex=float(\n                        np.nextafter(price_map[str(row_a["asset_id"])], math.inf)\n                    ).hex(),\n                ),\n            ),\n        ]:\n            ledger_path = dataset_dir / f"issue-{_safe_slug(label)}.sqlite"\n            if ledger_path.exists():\n                ledger_path.unlink()\n            ledger = DurableReleaseLedger(ledger_path)\n            store = RequestBoundSnapshotStore(\n                bundle.snapshot,\n                ledger=ledger,\n                asset_query_digests=bundle.query_digests,\n                secret=secrets.token_bytes(32),\n                verifier_tolerance=tolerance,\n            )\n            invalid_buffer = BoundAnswerBuffer(request=invalid_request, payload=answer_a)\n            try:\n                store.issue_token(\n                    bundle.snapshot,\n                    bundle.prices,\n                    invalid_request,\n                    invalid_buffer,\n                )\n                rejected = False\n                reason = "unexpected_token_issuance"\n            except ValueError as exc:\n                rejected = True\n                reason = str(exc)\n            fault_rows.append(\n                {\n                    "dataset": dataset,\n                    "case": label,\n                    "phase": "issuance",\n                    "expected_release": False,\n                    "released": False,\n                    "durable": False,\n                    "reason": reason,\n                    "buffer_released": invalid_buffer.released,\n                    "durable_release_rows": len(ledger.release_rows()),\n                    "passed": bool(rejected and len(ledger.release_rows()) == 0),\n                }\n            )\n\n        # Cross-service races exercise SQLite uniqueness and serialization, not\n        # only one store\'s in-process RLock.\n        schedules = {\n            "duplicate_token": "token_replay",\n            "distinct_tokens_same_request": "request_replay",\n        }\n        for schedule, expected_loser_reason in schedules.items():\n            ledger_path = dataset_dir / f"race-{schedule}.sqlite"\n            if ledger_path.exists():\n                ledger_path.unlink()\n            ledger_a = DurableReleaseLedger(ledger_path)\n            ledger_b = DurableReleaseLedger(ledger_path)\n            shared_secret = secrets.token_bytes(32)\n            store_a = RequestBoundSnapshotStore(\n                bundle.snapshot,\n                ledger=ledger_a,\n                asset_query_digests=bundle.query_digests,\n                secret=shared_secret,\n                verifier_tolerance=tolerance,\n            )\n            store_b = RequestBoundSnapshotStore(\n                bundle.snapshot,\n                ledger=ledger_b,\n                asset_query_digests=bundle.query_digests,\n                secret=shared_secret,\n                verifier_tolerance=tolerance,\n            )\n            for trial in range(int(concurrency_trials)):\n                race_request = _request_for_row(\n                    row_a,\n                    price_map,\n                    request_id=(\n                        f"{_safe_slug(dataset)}-{schedule}-race-{trial}"\n                    ),\n                )\n                if schedule == "duplicate_token":\n                    buffer_id = (\n                        f"{_safe_slug(dataset)}-duplicate-token-buffer-{trial}"\n                    )\n                    buffer_a = BoundAnswerBuffer(\n                        request=race_request,\n                        payload=answer_a,\n                        buffer_id=buffer_id,\n                    )\n                    buffer_b = BoundAnswerBuffer(\n                        request=race_request,\n                        payload=answer_a,\n                        buffer_id=buffer_id,\n                    )\n                    token_a = store_a.issue_token(\n                        bundle.snapshot, bundle.prices, race_request, buffer_a\n                    )\n                    token_b = token_a\n                else:\n                    buffer_a = BoundAnswerBuffer(\n                        request=race_request, payload=answer_a\n                    )\n                    buffer_b = BoundAnswerBuffer(\n                        request=race_request, payload=answer_a\n                    )\n                    token_a = store_a.issue_token(\n                        bundle.snapshot, bundle.prices, race_request, buffer_a\n                    )\n                    token_b = store_b.issue_token(\n                        bundle.snapshot, bundle.prices, race_request, buffer_b\n                    )\n\n                barrier = threading.Barrier(3)\n                result_holder: list[Any] = []\n                result_lock = threading.Lock()\n\n                def committer(\n                    store,\n                    candidate_buffer,\n                    candidate_token,\n                    ledger_instance,\n                ) -> None:\n                    barrier.wait(timeout=10)\n                    outcome = store.compare_and_commit(\n                        candidate_token,\n                        bundle.prices,\n                        race_request,\n                        candidate_buffer,\n                    )\n                    with result_lock:\n                        result_holder.append(outcome)\n                    ledger_instance.close_thread_connection()\n\n                threads = [\n                    threading.Thread(\n                        target=committer,\n                        args=(store_a, buffer_a, token_a, ledger_a),\n                    ),\n                    threading.Thread(\n                        target=committer,\n                        args=(store_b, buffer_b, token_b, ledger_b),\n                    ),\n                ]\n                for thread in threads:\n                    thread.start()\n                barrier.wait(timeout=10)\n                for thread in threads:\n                    thread.join(timeout=10)\n                releases = sum(bool(outcome.released) for outcome in result_holder)\n                ledger_rows = len(ledger_a.release_rows())\n                reasons = sorted(outcome.reason for outcome in result_holder)\n                passed = bool(\n                    len(result_holder) == 2\n                    and releases == 1\n                    and ledger_rows == trial + 1\n                    and reasons == ["released", expected_loser_reason]\n                )\n                concurrency_rows.append(\n                    {\n                        "dataset": dataset,\n                        "schedule": schedule,\n                        "trial": trial,\n                        "service_instances": 2,\n                        "completed_results": len(result_holder),\n                        "released_results": releases,\n                        "durable_release_rows": ledger_rows,\n                        "expected_durable_release_rows": trial + 1,\n                        "reasons": json.dumps(reasons),\n                        "passed": passed,\n                    }\n                )\n\n    return (\n        pd.DataFrame(fault_rows),\n        pd.DataFrame(concurrency_rows),\n        pd.DataFrame(delivery_rows),\n        pd.DataFrame(durable_rows),\n    )\n\n\ndef _candidate_digest(asset_ids: Sequence[str], candidate: np.ndarray) -> str:\n    return core.price_vector_digest(asset_ids, np.asarray(candidate, dtype=np.float64))\n\n\ndef _greedy_disjoint_rows(\n    snapshot: core.ConstraintSnapshot,\n    center: np.ndarray,\n    relation_family: str,\n    *,\n    tolerance: float,\n    budget_fraction: float,\n) -> tuple[list[list[tuple[str, int, tuple[int, ...]]]], np.ndarray, np.ndarray]:\n    scale = max(float(np.median(np.abs(center))), tolerance)\n    widths = float(budget_fraction) * np.maximum(np.abs(center), scale)\n    lower = np.maximum(0.0, center - widths)\n    upper = center + widths\n    attackable: list[tuple[str, int, tuple[int, ...]]] = []\n    for row, metadata in snapshot.row_metadata.iterrows():\n        if str(metadata["constraint_type"]) != relation_family:\n            continue\n        start, end = snapshot.matrix.indptr[row], snapshot.matrix.indptr[row + 1]\n        columns = snapshot.matrix.indices[start:end]\n        coefficients = snapshot.matrix.data[start:end]\n        extremal = np.where(coefficients >= 0.0, upper[columns], lower[columns])\n        residual = float(np.dot(coefficients, extremal) - snapshot.bounds[row])\n        if residual > tolerance:\n            attackable.append(\n                (str(metadata["row_id"]), int(row), tuple(int(column) for column in columns))\n            )\n    attackable.sort(key=lambda record: record[0])\n    pools: list[list[tuple[str, int, tuple[int, ...]]]] = []\n    for parity in [0, 1]:\n        used: set[int] = set()\n        selected = []\n        for index, record in enumerate(attackable):\n            if index % 2 != parity:\n                continue\n            if used.isdisjoint(record[2]):\n                selected.append(record)\n                used.update(record[2])\n        pools.append(selected)\n    return pools, lower, upper\n\n\ndef build_dense_candidate_catalog(\n    bundles: Mapping[str, DatasetBundle],\n    *,\n    tolerance: float,\n    target_violation_counts: Sequence[int] = (8, 192),\n    budget_fraction: float = 0.50,\n) -> tuple[list[dict[str, Any]], pd.DataFrame]:\n    """Create deterministic multi-row attacks with genuinely dense active sets."""\n\n    cases: list[dict[str, Any]] = []\n    public_rows: list[dict[str, Any]] = []\n    for dataset, bundle in bundles.items():\n        for family in RELATION_FAMILIES:\n            center, center_policy = relation_family_attack_center(\n                bundle.snapshot,\n                bundle.prices,\n                family,\n                tolerance=tolerance,\n            )\n            pools, lower, upper = _greedy_disjoint_rows(\n                bundle.snapshot,\n                center,\n                family,\n                tolerance=tolerance,\n                budget_fraction=budget_fraction,\n            )\n            for pool_index, target_count in enumerate(target_violation_counts):\n                pool = pools[pool_index % len(pools)]\n                search_limit = min(96, len(pool))\n                if search_limit < 2:\n                    raise RuntimeError(\n                        f"insufficient disjoint attack rows for {dataset}/{family}"\n                    )\n                candidates = []\n                for selected_count in range(1, search_limit + 1):\n                    candidate = center.copy()\n                    chosen = pool[:selected_count]\n                    for _, row, _ in chosen:\n                        start, end = (\n                            bundle.snapshot.matrix.indptr[row],\n                            bundle.snapshot.matrix.indptr[row + 1],\n                        )\n                        columns = bundle.snapshot.matrix.indices[start:end]\n                        coefficients = bundle.snapshot.matrix.data[start:end]\n                        candidate[columns] = np.where(\n                            coefficients >= 0.0,\n                            upper[columns],\n                            lower[columns],\n                        )\n                    verification = core.independent_verify(\n                        bundle.snapshot, candidate, tolerance=tolerance\n                    )\n                    candidates.append(\n                        (\n                            abs(math.log1p(verification.violations) - math.log1p(target_count)),\n                            selected_count,\n                            verification,\n                            candidate,\n                            chosen,\n                        )\n                    )\n                _, selected_count, verification, candidate, chosen = min(\n                    candidates, key=lambda record: (record[0], record[1])\n                )\n                case_id = hashlib.sha256(\n                    (\n                        f"{dataset}|{family}|{pool_index}|{target_count}|"\n                        f"{_candidate_digest(bundle.snapshot.asset_ids, candidate)}"\n                    ).encode("utf-8")\n                ).hexdigest()[:24]\n                case = {\n                    "dataset": dataset,\n                    "relation_family": family,\n                    "case_id": case_id,\n                    "target_violation_count": int(target_count),\n                    "selected_disjoint_rows": int(selected_count),\n                    "selected_row_ids": [record[0] for record in chosen],\n                    "violations_before": int(verification.violations),\n                    "max_residual_before": float(verification.max_residual),\n                    "candidate_digest": _candidate_digest(\n                        bundle.snapshot.asset_ids, candidate\n                    ),\n                    "center_policy": center_policy,\n                    "budget_fraction": float(budget_fraction),\n                    "_candidate": candidate,\n                    "_snapshot": bundle.snapshot,\n                }\n                cases.append(case)\n                public_rows.append(\n                    {\n                        key: (\n                            json.dumps(value)\n                            if key == "selected_row_ids"\n                            else value\n                        )\n                        for key, value in case.items()\n                        if not key.startswith("_")\n                    }\n                )\n    return cases, pd.DataFrame(public_rows)\n\n\ndef run_binding_budget_ablation(\n    cases: Sequence[dict[str, Any]],\n    *,\n    budgets: Sequence[int],\n    repetitions: int,\n    warmups: int,\n    tolerance: float,\n    max_sweeps: int,\n    step: float,\n    osqp_max_iter: int,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    rows: list[dict[str, Any]] = []\n    variants = [("active_only", False), ("fail_safe", True)]\n    for case_index, case in enumerate(cases):\n        snapshot = case["_snapshot"]\n        candidate = case["_candidate"]\n        precheck = core.independent_verify(snapshot, candidate, tolerance=tolerance)\n        if precheck.passed or precheck.violations != case["violations_before"]:\n            raise RuntimeError(f"dense candidate changed before ablation: {case[\'case_id\']}")\n        for budget in budgets:\n            if isinstance(budget, bool) or int(budget) != budget or int(budget) < 1:\n                raise ValueError("active-row budgets must be positive integers")\n            options = {\n                "max_sweeps": int(max_sweeps),\n                "active_row_budget": int(budget),\n                "step": float(step),\n                "osqp_max_iter": int(osqp_max_iter),\n            }\n            for _ in range(int(warmups)):\n                core.run_enforcement_method(\n                    snapshot,\n                    candidate,\n                    method="arbiter_active",\n                    tolerance=tolerance,\n                    allow_fallback=False,\n                    **options,\n                )\n                run_tiered_fail_safe_repair(\n                    snapshot,\n                    candidate,\n                    tolerance=tolerance,\n                    **options,\n                )\n            for repetition in range(int(repetitions)):\n                rotation = (case_index + int(budget) + repetition) % len(variants)\n                ordered_variants = variants[rotation:] + variants[:rotation]\n                for execution_order, (variant, allow_fallback) in enumerate(\n                    ordered_variants\n                ):\n                    if allow_fallback:\n                        result = run_tiered_fail_safe_repair(\n                            snapshot,\n                            candidate,\n                            tolerance=tolerance,\n                            **options,\n                        )\n                    else:\n                        result = core.run_enforcement_method(\n                            snapshot,\n                            candidate,\n                            method="arbiter_active",\n                            tolerance=tolerance,\n                            allow_fallback=False,\n                            **options,\n                        )\n                    oracle = core.independent_verify(\n                        snapshot, result.prices, tolerance=tolerance\n                    )\n                    rows.append(\n                        {\n                            "dataset": case["dataset"],\n                            "relation_family": case["relation_family"],\n                            "case_id": case["case_id"],\n                            "target_violation_count": case["target_violation_count"],\n                            "violations_before": precheck.violations,\n                            "budget_binding": bool(precheck.violations > int(budget)),\n                            "active_row_budget": int(budget),\n                            "max_sweeps": int(max_sweeps),\n                            "variant": variant,\n                            "repetition": repetition,\n                            "execution_order": execution_order,\n                            "runtime_ms": result.runtime_ms,\n                            "fallback_ms": result.fallback_ms,\n                            "active_path_ms": result.runtime_ms - result.fallback_ms,\n                            "fallback_used": result.fallback_used,\n                            "fallback_stage": getattr(\n                                result, "fallback_stage", "none"\n                            ),\n                            "rejected": result.rejected,\n                            "verified": result.verification.passed,\n                            "oracle_verified": oracle.passed,\n                            "violations_after": oracle.violations,\n                            "max_residual_after": oracle.max_residual,\n                            "unsafe_release": bool(\n                                not result.rejected and not oracle.passed\n                            ),\n                            "sweeps": result.sweeps,\n                            "projected_rows": result.projected_rows,\n                            "l1_adjustment": result.l1_adjustment,\n                            "solver": result.solver,\n                            "candidate_digest": case["candidate_digest"],\n                            "constraint_digest": snapshot.constraint_digest,\n                        }\n                    )\n    raw = pd.DataFrame(rows)\n    summary = (\n        raw.groupby(\n            [\n                "dataset",\n                "relation_family",\n                "case_id",\n                "violations_before",\n                "active_row_budget",\n                "budget_binding",\n                "variant",\n            ],\n            as_index=False,\n        )\n        .agg(\n            timing_repetitions=("runtime_ms", "size"),\n            runtime_p50_ms=("runtime_ms", "median"),\n            runtime_p95_ms=("runtime_ms", lambda values: float(np.quantile(values, 0.95))),\n            active_path_p50_ms=("active_path_ms", "median"),\n            fallback_rate=("fallback_used", "mean"),\n            rejection_rate=("rejected", "mean"),\n            verified_rate=("oracle_verified", "mean"),\n            unsafe_release_rate=("unsafe_release", "mean"),\n            projected_rows_median=("projected_rows", "median"),\n            sweeps_median=("sweeps", "median"),\n            l1_adjustment_median=("l1_adjustment", "median"),\n        )\n        .sort_values(\n            ["dataset", "relation_family", "case_id", "active_row_budget", "variant"]\n        )\n        .reset_index(drop=True)\n    )\n    return raw, summary\n\n\ndef _target_margin_candidate(\n    snapshot: core.ConstraintSnapshot,\n    center: np.ndarray,\n    family: str,\n    margin: float,\n) -> tuple[np.ndarray, int, str, float]:\n    family_rows = snapshot.row_metadata[\n        snapshot.row_metadata["constraint_type"].astype(str).eq(family)\n    ].sort_values("row_id")\n    if family_rows.empty:\n        raise RuntimeError(f"no {family} row for sensitivity campaign")\n    for row in family_rows.index:\n        start, end = snapshot.matrix.indptr[row], snapshot.matrix.indptr[row + 1]\n        columns = snapshot.matrix.indices[start:end]\n        coefficients = snapshot.matrix.data[start:end]\n        positive = np.flatnonzero(coefficients > 0.0)\n        if not positive.size:\n            continue\n        current_residual = float(\n            np.dot(coefficients, center[columns]) - snapshot.bounds[row]\n        )\n        coefficient_position = int(positive[0])\n        coefficient = float(coefficients[coefficient_position])\n        delta = (float(margin) - current_residual) / coefficient\n        if delta <= 0.0:\n            continue\n        candidate = center.copy()\n        candidate[int(columns[coefficient_position])] += delta\n        target_residual = float(\n            np.dot(coefficients, candidate[columns]) - snapshot.bounds[row]\n        )\n        if target_residual <= 0.0:\n            continue\n        return candidate, int(row), str(family_rows.loc[row, "row_id"]), target_residual\n    raise RuntimeError(f"could not construct a positive-margin {family} candidate")\n\n\ndef run_tolerance_scale_sensitivity(\n    bundles: Mapping[str, DatasetBundle],\n    *,\n    scale_factors: Sequence[float],\n    relative_tolerances: Sequence[float],\n    margin_ratios: Sequence[float],\n    fixed_absolute_tolerance: float,\n    max_sweeps: int,\n    active_row_budget: int,\n    step: float,\n    osqp_max_iter: int,\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    rows: list[dict[str, Any]] = []\n    fixed_rows: list[dict[str, Any]] = []\n    for dataset, bundle in bundles.items():\n        if not np.allclose(bundle.snapshot.bounds, 0.0, atol=0.0, rtol=0.0):\n            raise RuntimeError(\n                "scale sensitivity requires the audited homogeneous zero-bound snapshot"\n            )\n        for family in RELATION_FAMILIES:\n            center, center_policy = relation_family_attack_center(\n                bundle.snapshot,\n                bundle.prices,\n                family,\n                tolerance=1e-12,\n            )\n            positive = np.abs(center[np.isfinite(center) & (center > 0.0)])\n            if not positive.size:\n                raise RuntimeError(f"no positive prices for {dataset}/{family}")\n            raw_reference = float(np.quantile(positive, 0.95))\n            if not math.isfinite(raw_reference) or raw_reference <= 0.0:\n                raise RuntimeError("invalid p95 normalization reference")\n            normalized_center = center / raw_reference\n            base_check = core.independent_verify(\n                bundle.snapshot, normalized_center, tolerance=1e-12\n            )\n            if not base_check.passed:\n                raise RuntimeError(f"normalization broke feasibility for {dataset}/{family}")\n\n            for scale_factor in scale_factors:\n                reference_unit = float(scale_factor)\n                scaled_center = normalized_center * reference_unit\n                for margin_ratio in margin_ratios:\n                    margin = float(margin_ratio) * reference_unit\n                    candidate, target_row, target_row_id, target_residual = _target_margin_candidate(\n                        bundle.snapshot,\n                        scaled_center,\n                        family,\n                        margin,\n                    )\n                    candidate_digest = _candidate_digest(\n                        bundle.snapshot.asset_ids, candidate\n                    )\n                    fixed_check = core.independent_verify(\n                        bundle.snapshot,\n                        candidate,\n                        tolerance=fixed_absolute_tolerance,\n                    )\n                    fixed_rows.append(\n                        {\n                            "dataset": dataset,\n                            "relation_family": family,\n                            "center_policy": center_policy,\n                            "normalization": "p95_positive_price_equals_one",\n                            "raw_p95_reference": raw_reference,\n                            "price_scale_factor": reference_unit,\n                            "margin_ratio": float(margin_ratio),\n                            "target_row": target_row,\n                            "target_row_id": target_row_id,\n                            "target_residual": target_residual,\n                            "fixed_absolute_tolerance": float(fixed_absolute_tolerance),\n                            "violations": fixed_check.violations,\n                            "accepted": fixed_check.passed,\n                            "candidate_digest": candidate_digest,\n                        }\n                    )\n                    for relative_tolerance in relative_tolerances:\n                        absolute_tolerance = (\n                            float(relative_tolerance) * reference_unit\n                        )\n                        before = core.independent_verify(\n                            bundle.snapshot,\n                            candidate,\n                            tolerance=absolute_tolerance,\n                        )\n                        normalized_candidate = candidate / reference_unit\n                        normalized_before = core.independent_verify(\n                            bundle.snapshot,\n                            normalized_candidate,\n                            tolerance=float(relative_tolerance),\n                        )\n                        if (\n                            before.passed != normalized_before.passed\n                            or before.violations != normalized_before.violations\n                        ):\n                            raise RuntimeError(\n                                "raw and normalized pre-verification decisions diverged"\n                            )\n                        # Repair in declared p95-normalized price units. This\n                        # removes arbitrary currency magnitude from stopping\n                        # logic; the final raw-unit vector is still independently\n                        # verified under the signed absolute tolerance.\n                        result = core.run_enforcement_method(\n                            bundle.snapshot,\n                            normalized_candidate,\n                            method="arbiter_active",\n                            tolerance=float(relative_tolerance),\n                            max_sweeps=max_sweeps,\n                            active_row_budget=active_row_budget,\n                            step=step,\n                            allow_fallback=True,\n                            osqp_max_iter=osqp_max_iter,\n                        )\n                        repaired_raw = result.prices * reference_unit\n                        oracle = core.independent_verify(\n                            bundle.snapshot,\n                            repaired_raw,\n                            tolerance=absolute_tolerance,\n                        )\n                        rows.append(\n                            {\n                                "dataset": dataset,\n                                "relation_family": family,\n                                "center_policy": center_policy,\n                                "normalization": "p95_positive_price_equals_one",\n                                "repair_space": "p95_normalized",\n                                "raw_p95_reference": raw_reference,\n                                "price_scale_factor": reference_unit,\n                                "reference_unit": reference_unit,\n                                "margin_ratio": float(margin_ratio),\n                                "relative_tolerance": float(relative_tolerance),\n                                "absolute_tolerance": absolute_tolerance,\n                                "target_row": target_row,\n                                "target_row_id": target_row_id,\n                                "target_residual": target_residual,\n                                "target_residual_ratio": target_residual / reference_unit,\n                                "violations_before": before.violations,\n                                "accepted_without_repair": before.passed,\n                                "verified_after": oracle.passed,\n                                "rejected": result.rejected,\n                                "fallback_used": result.fallback_used,\n                                "unsafe_release": bool(\n                                    not result.rejected and not oracle.passed\n                                ),\n                                "violations_after": oracle.violations,\n                                "max_residual_after": oracle.max_residual,\n                                "runtime_ms": result.runtime_ms,\n                                "l1_adjustment": float(\n                                    np.abs(repaired_raw - candidate).sum()\n                                ),\n                                "normalized_l1_adjustment": result.l1_adjustment,\n                                "candidate_digest": candidate_digest,\n                                "constraint_digest": bundle.snapshot.constraint_digest,\n                            }\n                        )\n\n    raw = pd.DataFrame(rows)\n    fixed = pd.DataFrame(fixed_rows)\n    invariance_rows = []\n    group_keys = [\n        "dataset",\n        "relation_family",\n        "margin_ratio",\n        "relative_tolerance",\n    ]\n    for keys, group in raw.groupby(group_keys, sort=True):\n        violations_invariant = group["violations_before"].nunique() == 1\n        acceptance_invariant = group["accepted_without_repair"].nunique() == 1\n        release_invariant = group["rejected"].nunique() == 1\n        distortion = group["normalized_l1_adjustment"].to_numpy(float)\n        distortion_span = float(distortion.max() - distortion.min())\n        distortion_scale = max(float(np.abs(distortion).max()), 1e-15)\n        distortion_relative_span = distortion_span / distortion_scale\n        # A percentage is unstable when the adjustment itself is near zero.\n        # Judge scale invariance in normalized price units, with a bound tied to\n        # the configured verifier tolerance, while retaining the relative span\n        # as a diagnostic column.\n        distortion_span_limit = max(1e-9, 5.0 * float(keys[3]))\n        invariance_rows.append(\n            {\n                **dict(zip(group_keys, keys, strict=True)),\n                "scale_points": int(group["price_scale_factor"].nunique()),\n                "violations_invariant": violations_invariant,\n                "acceptance_invariant": acceptance_invariant,\n                "release_invariant": release_invariant,\n                "normalized_distortion_span": distortion_span,\n                "normalized_distortion_relative_span": distortion_relative_span,\n                "normalized_distortion_span_limit": distortion_span_limit,\n                "passed": bool(\n                    violations_invariant\n                    and acceptance_invariant\n                    and release_invariant\n                    and distortion_span <= distortion_span_limit\n                    and group["verified_after"].all()\n                    and not group["unsafe_release"].any()\n                ),\n            }\n        )\n    invariance = pd.DataFrame(invariance_rows)\n\n    fixed_drift_rows = []\n    fixed_keys = ["dataset", "relation_family", "margin_ratio"]\n    for keys, group in fixed.groupby(fixed_keys, sort=True):\n        fixed_drift_rows.append(\n            {\n                **dict(zip(fixed_keys, keys, strict=True)),\n                "scale_points": int(group["price_scale_factor"].nunique()),\n                "distinct_acceptance_decisions": int(group["accepted"].nunique()),\n                "decision_drift_across_scale": bool(group["accepted"].nunique() > 1),\n            }\n        )\n    fixed_drift = pd.DataFrame(fixed_drift_rows)\n    return raw, invariance, fixed, fixed_drift\n\n\ndef run_protocol_overhead_benchmark(\n    bundles: Mapping[str, DatasetBundle],\n    *,\n    answer_executor: Callable[[str, str], Any],\n    work_dir: Path,\n    tolerance: float,\n    repetitions: int,\n    warmups: int,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Separate answer hashing/staging, authorization, commit, and delivery costs."""\n\n    work_dir.mkdir(parents=True, exist_ok=True)\n    rows: list[dict[str, Any]] = []\n    for dataset, bundle in bundles.items():\n        row_a, _ = _protocol_assets(bundle)\n        answer = answer_executor(dataset, str(row_a["sql"]))\n        price_map = dict(zip(bundle.snapshot.asset_ids, bundle.prices, strict=True))\n        ledger_path = work_dir / f"overhead-{_safe_slug(dataset)}.sqlite"\n        if ledger_path.exists():\n            ledger_path.unlink()\n        ledger = DurableReleaseLedger(ledger_path)\n        store = RequestBoundSnapshotStore(\n            bundle.snapshot,\n            ledger=ledger,\n            asset_query_digests=bundle.query_digests,\n            secret=secrets.token_bytes(32),\n            verifier_tolerance=tolerance,\n        )\n\n        def one(repetition: int, measured: bool) -> None:\n            request = _request_for_row(\n                row_a,\n                price_map,\n                request_id=f"overhead-{_safe_slug(dataset)}-{repetition}-{\'m\' if measured else \'w\'}",\n            )\n            started = time.perf_counter_ns()\n            buffer = BoundAnswerBuffer(request=request, payload=answer)\n            buffer_ms = (time.perf_counter_ns() - started) / 1e6\n\n            started = time.perf_counter_ns()\n            ledger.stage_buffer(request, buffer)\n            stage_ms = (time.perf_counter_ns() - started) / 1e6\n\n            started = time.perf_counter_ns()\n            token = store.issue_token(\n                bundle.snapshot, bundle.prices, request, buffer\n            )\n            issue_ms = (time.perf_counter_ns() - started) / 1e6\n\n            started = time.perf_counter_ns()\n            commit = store.compare_and_commit(\n                token, bundle.prices, request, buffer\n            )\n            commit_ms = (time.perf_counter_ns() - started) / 1e6\n\n            started = time.perf_counter_ns()\n            delivered = ledger.released_payload(\n                token_id=token.token_id, request=request\n            )\n            delivery_ms = (time.perf_counter_ns() - started) / 1e6\n            if not commit.released or delivered != buffer.payload_bytes:\n                raise RuntimeError("protocol overhead benchmark failed its release check")\n            if measured:\n                rows.append(\n                    {\n                        "dataset": dataset,\n                        "repetition": repetition,\n                        "payload_bytes": len(buffer.payload_bytes),\n                        "buffer_digest_ms": buffer_ms,\n                        "durable_stage_ms": stage_ms,\n                        "token_issue_ms": issue_ms,\n                        "durable_commit_ms": commit_ms,\n                        "delivery_integrity_ms": delivery_ms,\n                        "authorization_ms": issue_ms + commit_ms,\n                        "control_without_delivery_ms": stage_ms + issue_ms + commit_ms,\n                        "complete_buffer_to_delivery_ms": (\n                            buffer_ms + stage_ms + issue_ms + commit_ms + delivery_ms\n                        ),\n                        "released": commit.released,\n                        "durable": commit.durable,\n                    }\n                )\n\n        for warmup in range(int(warmups)):\n            one(-warmup - 1, False)\n        for repetition in range(int(repetitions)):\n            one(repetition, True)\n\n    raw = pd.DataFrame(rows)\n    summary = (\n        raw.groupby("dataset", as_index=False)\n        .agg(\n            timing_repetitions=("repetition", "size"),\n            payload_bytes=("payload_bytes", "first"),\n            buffer_digest_p50_ms=("buffer_digest_ms", "median"),\n            stage_p50_ms=("durable_stage_ms", "median"),\n            issue_p50_ms=("token_issue_ms", "median"),\n            commit_p50_ms=("durable_commit_ms", "median"),\n            delivery_p50_ms=("delivery_integrity_ms", "median"),\n            authorization_p50_ms=("authorization_ms", "median"),\n            complete_p50_ms=("complete_buffer_to_delivery_ms", "median"),\n            complete_p95_ms=(\n                "complete_buffer_to_delivery_ms",\n                lambda values: float(np.quantile(values, 0.95)),\n            ),\n            release_rate=("released", "mean"),\n            durable_rate=("durable", "mean"),\n        )\n        .sort_values("dataset")\n        .reset_index(drop=True)\n    )\n    return raw, summary\n\n\ndef hardware_environment_manifest(extra: Mapping[str, Any] | None = None) -> dict[str, Any]:\n    cpu_model = "unknown"\n    cpuinfo = Path("/proc/cpuinfo")\n    if cpuinfo.is_file():\n        for line in cpuinfo.read_text(encoding="utf-8", errors="replace").splitlines():\n            if line.lower().startswith("model name") and ":" in line:\n                cpu_model = line.split(":", 1)[1].strip()\n                break\n    memory_bytes = None\n    meminfo = Path("/proc/meminfo")\n    if meminfo.is_file():\n        for line in meminfo.read_text(encoding="utf-8", errors="replace").splitlines():\n            if line.startswith("MemTotal:"):\n                memory_bytes = int(line.split()[1]) * 1024\n                break\n    clock = time.get_clock_info("perf_counter")\n    result = core.environment_manifest(extra=extra)\n    result.update(\n        {\n            "closure_suite_version": CLOSURE_SUITE_VERSION,\n            "cpu_model": cpu_model,\n            "logical_cpu_count": os.cpu_count(),\n            "memory_bytes": memory_bytes,\n            "machine": platform.machine(),\n            "perf_counter_resolution_seconds": clock.resolution,\n            "perf_counter_monotonic": clock.monotonic,\n            "sqlite_version": __import__("sqlite3").sqlite_version,\n            "thread_environment": {\n                name: os.environ.get(name, "unset")\n                for name in [\n                    "OMP_NUM_THREADS",\n                    "OPENBLAS_NUM_THREADS",\n                    "MKL_NUM_THREADS",\n                    "NUMEXPR_NUM_THREADS",\n                ]\n            },\n        }\n    )\n    return result\n\n\ndef closure_quality_checks(\n    *,\n    input_audit: pd.DataFrame,\n    protocol_faults: pd.DataFrame,\n    concurrency: pd.DataFrame,\n    delivery: pd.DataFrame,\n    budget_raw: pd.DataFrame,\n    tolerance_raw: pd.DataFrame,\n    invariance: pd.DataFrame,\n    fixed_drift: pd.DataFrame,\n    overhead_raw: pd.DataFrame,\n) -> pd.DataFrame:\n    checks = [\n        ("all audited-v4 input hashes and code hashes match", input_audit["passed"].all()),\n        ("every protocol substitution and issuance fault passes", protocol_faults["passed"].all()),\n        ("authorized and unauthorized deliveries behave correctly", delivery["passed"].all()),\n        ("cross-service duplicate-token and same-request races allow exactly one release", concurrency["passed"].all()),\n        ("every tested active-row budget is binding in at least one case", all(\n            budget_raw.loc[budget_raw["active_row_budget"] == budget, "budget_binding"].any()\n            for budget in sorted(budget_raw["active_row_budget"].unique())\n        )),\n        ("dense ablation contains violations above the former observed maximum", budget_raw["violations_before"].max() > 47),\n        ("active-only ablation exercises rejection", budget_raw.loc[budget_raw["variant"] == "active_only", "rejected"].any()),\n        ("fail-safe ablation exercises fallback", budget_raw.loc[budget_raw["variant"] == "fail_safe", "fallback_used"].any()),\n        ("fail-safe ablation releases no unverified vector", not budget_raw.loc[budget_raw["variant"] == "fail_safe", "unsafe_release"].any()),\n        ("scale-aware tolerance decisions and normalized distortion are invariant", invariance["passed"].all()),\n        ("fixed absolute tolerance diagnostic exposes at least one scale-dependent decision", fixed_drift["decision_drift_across_scale"].any()),\n        ("all scale-aware repaired outputs independently verify", tolerance_raw["verified_after"].all()),\n        ("tolerance campaign records no unsafe release", not tolerance_raw["unsafe_release"].any()),\n        ("protocol overhead trials all durably release", overhead_raw["released"].all() and overhead_raw["durable"].all()),\n    ]\n    return pd.DataFrame(\n        [{"check": label, "passed": bool(passed)} for label, passed in checks]\n    )\n',
}
expected_embedded_hashes = {
    "arbiter_request_release.py": "b00dc16f4ab5ea9fc57d30fb141c83f31003bc8014e8285c1e1eabd812f66fc3",
    "arbiter_closure_experiments.py": "bf63760ee2124e6976bcf02e17d14c00c083defa4bdedae70fedfdec46e7cfb4",
}
for name, module_source in embedded_sources.items():
    module_path = local_code_dir / name
    module_path.write_text(module_source, encoding="utf-8")
    observed = hashlib.sha256(module_path.read_bytes()).hexdigest()
    if observed != expected_embedded_hashes[name]:
        raise RuntimeError(f"Embedded module hash mismatch: {name}")

for path in [str(local_code_dir), str(CODE_DIR.resolve())]:
    if path in sys.path:
        sys.path.remove(path)
sys.path.insert(0, str(local_code_dir))
sys.path.insert(1, str(CODE_DIR.resolve()))
for module_name in [
    "arbiter_request_release",
    "arbiter_closure_experiments",
    "arbiter_core",
    "arbiter_experiments",
    "arbiter_workload",
    "arbiter_data",
]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import duckdb
import numpy as np
import pandas as pd

import arbiter_core as core
from arbiter_closure_experiments import (
    AUDITED_CODE_HASHES,
    CLOSURE_SUITE_VERSION,
    build_dataset_bundles,
    build_dense_candidate_catalog,
    closure_quality_checks,
    hardware_environment_manifest,
    run_binding_budget_ablation,
    run_protocol_fault_campaign,
    run_protocol_overhead_benchmark,
    run_tolerance_scale_sensitivity,
    sha256_file,
    verify_audited_inputs,
)
from arbiter_request_release import RequestBoundReleaseToken

print("Startup import self-check PASS")
print("Active audited run:", ACTIVE_RUN_DIR)
print("Validated evidence package:", EVIDENCE_DIR)
print("Evidence package source:", EVIDENCE_SOURCE)
print("Closure mode:", CLOSURE_MODE)

# Revalidate every packaged evidence file, the exact audited source bytes,
# and the equality of the nested package with the live 00--04 stage files.
input_audit = verify_audited_inputs(
    EVIDENCE_DIR,
    CODE_DIR,
    active_run_dir=ACTIVE_RUN_DIR,
)
print(f"Audited input integrity PASS ({len(input_audit)} checks)")

run_config = json.loads(
    (EVIDENCE_DIR / "run_config.json").read_text(encoding="utf-8")
)
tolerance = float(run_config["tolerance"])
assets = pd.read_csv(EVIDENCE_DIR / "00_workload" / "sql_assets.csv")
relations = pd.read_csv(
    EVIDENCE_DIR / "00_workload" / "compiled_relations.csv"
)
policy = pd.read_csv(
    EVIDENCE_DIR / "01_proposals_attacks" / "controlled_policy.csv"
)
bundles = build_dataset_bundles(
    assets,
    relations,
    policy,
    tolerance=tolerance,
)

database_path = (
    PROJECT_ROOT
    / "state"
    / f"arbiter_sf{run_config['tpch_scale_factor']:g}.duckdb"
)
if not database_path.is_file():
    raise FileNotFoundError(
        "The audited DuckDB database is missing. Rerun audited Notebook 00; "
        "this notebook has no synthetic-answer fallback."
    )
connection = duckdb.connect(str(database_path), read_only=True)

answer_provenance_rows = []
def execute_actual_answer(dataset, sql):
    started = time.perf_counter_ns()
    answer = connection.execute(sql).fetchall()
    elapsed_ms = (time.perf_counter_ns() - started) / 1e6
    answer_provenance_rows.append(
        {
            "dataset": dataset,
            "sql_sha256": hashlib.sha256(sql.encode("utf-8")).hexdigest(),
            "answer_rows": len(answer),
            "query_ms": elapsed_ms,
            "source": "DuckDB execution over audited NYC/Adult/TPC-H database",
        }
    )
    return answer

label_slug = re.sub(
    r"[^a-z0-9]+", "-", ENVIRONMENT_LABEL.lower()
).strip("-")
if not label_slug:
    raise ValueError("ENVIRONMENT_LABEL must contain letters or digits.")
output_name = f"05_request_bound_closure_{CLOSURE_MODE}_{label_slug}"
output_dir = (ACTIVE_RUN_DIR / output_name).resolve()
if ACTIVE_RUN_DIR not in output_dir.parents:
    raise RuntimeError("Closure output path escaped the audited run directory.")
if output_dir.exists():
    shutil.rmtree(output_dir)
for relative in ["protocol", "budget", "sensitivity", "performance", "code"]:
    (output_dir / relative).mkdir(parents=True, exist_ok=True)

closure_config = {
    "closure_suite_version": CLOSURE_SUITE_VERSION,
    "closure_mode": CLOSURE_MODE,
    "environment_label": ENVIRONMENT_LABEL,
    "audited_input_suite_version": run_config["suite_version"],
    "audited_input_run_mode": run_config["run_mode"],
    "verifier_tolerance": tolerance,
    "active_row_budgets": [1, 4, 8, 16, 32, 64],
    "dense_target_violation_counts": [8, 192],
    "scale_factors": [1e-2, 1.0, 1e2],
    "fixed_absolute_tolerance_diagnostic": tolerance,
    "max_sweeps": int(run_config["repair_sweeps"]),
    "repair_step": float(run_config["repair_step"]),
    "osqp_max_iter": int(run_config["osqp_max_iter"]),
    **mode_options,
}
(output_dir / "closure_run_config.json").write_text(
    json.dumps(closure_config, indent=2, sort_keys=True),
    encoding="utf-8",
)

with tempfile.TemporaryDirectory(
    prefix="arbiter-request-bound-closure-work-"
) as temporary_work:
    temporary_work = Path(temporary_work)
    protocol_faults, concurrency, delivery, durable_records = (
        run_protocol_fault_campaign(
            bundles,
            answer_executor=execute_actual_answer,
            work_dir=temporary_work / "protocol",
            tolerance=tolerance,
            concurrency_trials=mode_options["concurrency_trials"],
        )
    )
    overhead_raw, overhead_summary = run_protocol_overhead_benchmark(
        bundles,
        answer_executor=execute_actual_answer,
        work_dir=temporary_work / "overhead",
        tolerance=tolerance,
        repetitions=mode_options["overhead_repetitions"],
        warmups=mode_options["overhead_warmups"],
    )

dense_cases, dense_catalog = build_dense_candidate_catalog(
    bundles,
    tolerance=tolerance,
    target_violation_counts=(8, 192),
    budget_fraction=0.50,
)
budget_raw, budget_summary = run_binding_budget_ablation(
    dense_cases,
    budgets=[1, 4, 8, 16, 32, 64],
    repetitions=mode_options["budget_repetitions"],
    warmups=mode_options["budget_warmups"],
    tolerance=tolerance,
    max_sweeps=int(run_config["repair_sweeps"]),
    step=float(run_config["repair_step"]),
    osqp_max_iter=int(run_config["osqp_max_iter"]),
)

tolerance_raw, invariance, fixed_absolute, fixed_drift = (
    run_tolerance_scale_sensitivity(
        bundles,
        scale_factors=[1e-2, 1.0, 1e2],
        relative_tolerances=mode_options["relative_tolerances"],
        margin_ratios=mode_options["margin_ratios"],
        fixed_absolute_tolerance=tolerance,
        max_sweeps=int(run_config["repair_sweeps"]),
        active_row_budget=64,
        step=float(run_config["repair_step"]),
        osqp_max_iter=int(run_config["osqp_max_iter"]),
    )
)

input_audit.to_csv(output_dir / "input_evidence_audit.csv", index=False)
protocol_faults.to_csv(
    output_dir / "protocol" / "request_binding_faults.csv", index=False
)
concurrency.to_csv(
    output_dir / "protocol" / "cross_service_concurrency.csv", index=False
)
delivery.to_csv(
    output_dir / "protocol" / "delivery_authorization.csv", index=False
)
durable_records.to_csv(
    output_dir / "protocol" / "durable_release_records.csv", index=False
)
pd.DataFrame(answer_provenance_rows).to_csv(
    output_dir / "protocol" / "answer_provenance.csv", index=False
)
dense_catalog.to_csv(
    output_dir / "budget" / "dense_candidate_catalog.csv", index=False
)
budget_raw.to_csv(
    output_dir / "budget" / "active_budget_trials_raw.csv", index=False
)
budget_summary.to_csv(
    output_dir / "budget" / "active_budget_summary.csv", index=False
)
tolerance_raw.to_csv(
    output_dir / "sensitivity" / "scale_aware_tolerance_raw.csv", index=False
)
invariance.to_csv(
    output_dir / "sensitivity" / "scale_invariance_checks.csv", index=False
)
fixed_absolute.to_csv(
    output_dir / "sensitivity" / "fixed_absolute_diagnostic_raw.csv", index=False
)
fixed_drift.to_csv(
    output_dir / "sensitivity" / "fixed_absolute_scale_drift.csv", index=False
)
overhead_raw.to_csv(
    output_dir / "performance" / "protocol_overhead_raw.csv", index=False
)
overhead_summary.to_csv(
    output_dir / "performance" / "protocol_overhead_summary.csv", index=False
)

quality = closure_quality_checks(
    input_audit=input_audit,
    protocol_faults=protocol_faults,
    concurrency=concurrency,
    delivery=delivery,
    budget_raw=budget_raw,
    tolerance_raw=tolerance_raw,
    invariance=invariance,
    fixed_drift=fixed_drift,
    overhead_raw=overhead_raw,
)

required_token_fields = {
    "request_id", "buyer_id", "session_id", "asset_id",
    "query_digest", "quoted_price_hex", "request_digest",
    "buffer_id", "answer_digest", "price_digest",
    "constraint_digest", "asset_digest", "epoch",
}
actual_token_fields = set(RequestBoundReleaseToken.__dataclass_fields__)
structural_checks = pd.DataFrame(
    [
        {
            "check": "request-bound token exposes every required signed field",
            "passed": required_token_fields.issubset(actual_token_fields),
        },
        {
            "check": "protocol faults have unique dataset/case keys",
            "passed": not protocol_faults.duplicated(["dataset", "case"]).any(),
        },
        {
            "check": "concurrency rows have unique dataset/schedule/trial keys",
            "passed": not concurrency.duplicated(["dataset", "schedule", "trial"]).any(),
        },
        {
            "check": "budget timing rows have unique paired keys",
            "passed": not budget_raw.duplicated(
                ["dataset", "relation_family", "case_id", "active_row_budget", "variant", "repetition"]
            ).any(),
        },
        {
            "check": "tolerance grid rows have unique condition keys",
            "passed": not tolerance_raw.duplicated(
                ["dataset", "relation_family", "price_scale_factor", "margin_ratio", "relative_tolerance"]
            ).any(),
        },
        {
            "check": "actual DuckDB answers were used without fallback",
            "passed": bool(answer_provenance_rows)
            and set(pd.DataFrame(answer_provenance_rows)["source"])
            == {"DuckDB execution over audited NYC/Adult/TPC-H database"},
        },
    ]
)
quality = pd.concat([quality, structural_checks], ignore_index=True)

numeric_frames = {
    "protocol_faults": protocol_faults,
    "concurrency": concurrency,
    "budget_raw": budget_raw,
    "tolerance_raw": tolerance_raw,
    "overhead_raw": overhead_raw,
}
finite_rows = []
for frame_name, frame in numeric_frames.items():
    numeric = frame.select_dtypes(include=[np.number])
    finite_rows.append(
        {
            "check": f"{frame_name} contains no NaN or infinite numeric evidence",
            "passed": bool(np.isfinite(numeric.to_numpy(dtype=float)).all()),
        }
    )
quality = pd.concat([quality, pd.DataFrame(finite_rows)], ignore_index=True)
quality.to_csv(output_dir / "stage_quality_checks.csv", index=False)

claim_rows = [
    {
        "claim": "Request-, buyer/session-, asset-, query-, quote-, vector-, snapshot-, and answer-bound durable outbox commit for the evaluated prototype faults.",
        "status": "SUPPORTED" if protocol_faults["passed"].all() and concurrency["passed"].all() else "NOT_SUPPORTED",
        "required_wording": "Limit to the evaluated in-process service plus local SQLite transaction.",
    },
    {
        "claim": "The 1/4/8/16/32/64 active-row budgets are exercised by dense controlled cases.",
        "status": "SUPPORTED" if all(
            budget_raw.loc[budget_raw["active_row_budget"] == budget, "budget_binding"].any()
            for budget in [1, 4, 8, 16, 32, 64]
        ) else "NOT_SUPPORTED",
        "required_wording": "Call these deterministic constraint-level stress cases, not observed buyer attacks.",
    },
    {
        "claim": "P95-normalized repair is decision-invariant across the tested 1e-2/1/1e2 unit rescalings.",
        "status": "SUPPORTED" if invariance["passed"].all() else "NOT_SUPPORTED",
        "required_wording": "Report the tested relative-tolerance and residual-margin grid.",
    },
    {
        "claim": "Network delivery, distributed transactions, and payment settlement are atomic.",
        "status": "NOT_SUPPORTED",
        "required_wording": "The durable outbox makes delivery recoverable; it does not prove network delivery or payment atomicity.",
    },
    {
        "claim": "General SQL or standard TPC-H benchmark-query coverage.",
        "status": "NOT_SUPPORTED",
        "required_wording": "Retain the audited supported-fragment boundary.",
    },
    {
        "claim": "Performance generalizes across hardware environments.",
        "status": "NOT_SUPPORTED",
        "required_wording": "This archive records one named environment; a second environment remains required.",
    },
]
pd.DataFrame(claim_rows).to_csv(output_dir / "claim_audit.csv", index=False)

environment = hardware_environment_manifest(extra=closure_config)
(output_dir / "environment.json").write_text(
    json.dumps(environment, indent=2, sort_keys=True), encoding="utf-8"
)
for name, module_source in embedded_sources.items():
    (output_dir / "code" / name).write_text(module_source, encoding="utf-8")
(output_dir / "code" / "code_manifest.json").write_text(
    json.dumps(expected_embedded_hashes, indent=2, sort_keys=True),
    encoding="utf-8",
)

readme = f"""# ARBITER request-bound closure {CLOSURE_MODE} run

This targeted campaign preserves the audited-v4 00-04 evidence and evaluates
only the closure layer. It uses actual DuckDB query answers over the already
audited NYC Taxi, UCI Adult, and TPC-H SF1 database. It has no synthetic-data or
synthetic-answer fallback. Prices, dense attacks, and tolerance margins are
controlled deterministic experimental constructions, not market observations.

The durable claim ends at a local SQLite transaction that records the purchase
authorization and moves the staged answer into a released outbox. Network
delivery, distributed transactions, and payment settlement are not claimed.

Environment label: {ENVIRONMENT_LABEL}
Closure mode: {CLOSURE_MODE}
Active run: {ACTIVE_RUN_DIR.name}
Audited evidence package: {EVIDENCE_DIR.name}
Evidence source: {EVIDENCE_SOURCE}
"""
(output_dir / "README.md").write_text(readme, encoding="utf-8")

if not quality["passed"].all():
    display(quality)
    failed = quality.loc[~quality["passed"], "check"].tolist()
    raise RuntimeError(
        "Closure quality checks failed. Do not use these results: "
        + "; ".join(failed)
    )

# Recompute output checksums after all evidence is final.
checksum_rows = []
for path in sorted(output_dir.rglob("*")):
    if path.is_file() and path.name not in {"checksums.csv", "checksums.sha256"}:
        checksum_rows.append(
            {
                "path": path.relative_to(output_dir).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
checksum_frame = pd.DataFrame(checksum_rows)
checksum_frame.to_csv(output_dir / "checksums.csv", index=False)
(output_dir / "checksums.sha256").write_text(
    "".join(
        f"{row.sha256}  {row.path}\n"
        for row in checksum_frame.itertuples(index=False)
    ),
    encoding="utf-8",
)
for row in checksum_frame.itertuples(index=False):
    target = output_dir / row.path
    if target.stat().st_size != int(row.bytes) or sha256_file(target) != row.sha256:
        raise RuntimeError(f"Post-write checksum verification failed: {row.path}")

archive_path = ACTIVE_RUN_DIR / f"arbiter_icde_closure_{CLOSURE_MODE}_{label_slug}_v1_1.zip"
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9
) as archive:
    for path in sorted(output_dir.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                (Path(output_dir.name) / path.relative_to(output_dir)).as_posix(),
            )
with zipfile.ZipFile(archive_path) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP verification failed at {bad_member}")

connection.close()
display(quality)
display(
    budget_summary.groupby(["active_row_budget", "variant"], as_index=False)
    .agg(
        cases=("case_id", "nunique"),
        fallback_rate=("fallback_rate", "mean"),
        rejection_rate=("rejection_rate", "mean"),
        runtime_p50_ms=("runtime_p50_ms", "median"),
    )
)
display(overhead_summary)
print("Closure notebook PASS")
print("Return this ZIP for independent audit:")
print(archive_path)


Mounted at /content/drive
Startup import self-check PASS
Active audited run: /content/drive/MyDrive/Arbiter_ICDE_2027/runs/full_audited_v4
Validated evidence package: /content/drive/MyDrive/Arbiter_ICDE_2027/runs/full_audited_v4/arbiter_icde_evidence_full_audited_v4
Evidence package source: nested_notebook04_package
Closure mode: full
Audited input integrity PASS (124 checks)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,check,passed
0,all audited-v4 input hashes and code hashes match,True
1,every protocol substitution and issuance fault...,True
2,authorized and unauthorized deliveries behave ...,True
3,cross-service duplicate-token and same-request...,True
4,every tested active-row budget is binding in a...,True
5,dense ablation contains violations above the f...,True
6,active-only ablation exercises rejection,True
7,fail-safe ablation exercises fallback,True
8,fail-safe ablation releases no unverified vector,True
9,scale-aware tolerance decisions and normalized...,True


,active_row_budget,variant,cases,fallback_rate,rejection_rate,runtime_p50_ms
0,1,active_only,18,0.000000,0.166667,1.879031
1,1,fail_safe,18,0.166667,0.000000,1.849398
2,4,active_only,18,0.000000,0.055556,1.195283
3,4,fail_safe,18,0.055556,0.000000,1.213356
4,8,active_only,18,0.000000,0.000000,1.141606
5,8,fail_safe,18,0.000000,0.000000,1.131239
6,16,active_only,18,0.000000,0.000000,1.157964
7,16,fail_safe,18,0.000000,0.000000,1.143090
8,32,active_only,18,0.000000,0.000000,1.129714
9,32,fail_safe,18,0.000000,0.000000,1.149360


,dataset,timing_repetitions,payload_bytes,buffer_digest_p50_ms,stage_p50_ms,issue_p50_ms,commit_p50_ms,delivery_p50_ms,authorization_p50_ms,complete_p50_ms,complete_p95_ms,release_rate,durable_rate
0,NYC Taxi 2024-01,25,85,0.074605,4.564487,2.849771,9.291541,0.149156,12.030422,16.990083,18.722151,1.0,1.0
1,TPC-H SF1,25,70,0.081384,4.580747,2.909098,9.333748,0.148936,12.413878,17.501932,20.740165,1.0,1.0
2,UCI Adult,25,67,0.072258,4.686650,2.868749,9.139348,0.144312,11.961771,17.163071,18.844980,1.0,1.0


Closure notebook PASS
Return this ZIP for independent audit:
/content/drive/MyDrive/Arbiter_ICDE_2027/runs/full_audited_v4/arbiter_icde_closure_full_colab-cpu-primary_v1_1.zip
